# Лекция 3. Память LLM-систем

*Искусственный интеллект в инженерном образовании · НИУ ВШЭ · 2026/27*

Знакомьтесь: Паша. Бухгалтер, играет на гитаре, три месяца болтает со своим ИИ-ассистентом: то рецепт, то письмо
подрядчику, то какую сковородку купить. Почти в каждом разговоре курица: в сливочном соусе, с фасолью, с картошкой.
А неделю назад Паша между делом написал: «окончательно перешёл на вегетарианство, мясо и рыбу совсем исключил».

Сегодня вечером Паша спросит у ассистента, что приготовить на ужин. Угадайте, что ему посоветуют.

Это лекция о том, почему у языковой модели **нет памяти**, как её приделать снаружи и как проверить, что приделали
хорошо. По дороге мы:
1. убедимся, что модель ничего не помнит между вызовами, и посчитаем, сколько стоит «помнить всё»;
2. выясним, почему окно в 262 тысячи токенов не решает задачу;
3. соберём десяток вариантов памяти: от буфера до извлечённых фактов с консолидацией, как в Mem0, и устроим им турнир
   на сотне вопросов;
4. дадим модели самой управлять своей памятью (MemGPT / Letta);
5. заглянем в память *внутри* архитектуры Qwen3.5 и напишем её с нуля;
6. научим модель помнить **видео**, которое не влезает ни в какой контекст;
7. и выполним просьбу «удалите всё, что вы обо мне знаете». Это сложнее, чем кажется.

**Как запускать.** Colab → *Среда выполнения → Сменить среду → T4 GPU* → ячейки по порядку. Ключи не нужны.

## 0. Подготовка

In [ ]:
import sys, os, time, json, re, math, html, random, subprocess, base64, gzip, collections, copy
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install langchain==1.4.6 langchain-ollama==1.1.0 transformers==5.20.2 pydantic==2.14.1
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], env=dict(os.environ, OLLAMA_KEEP_ALIVE="-1", OLLAMA_NUM_PARALLEL="2"),
                     stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(5)

In [ ]:
MODEL = os.environ.get("LECTURE_MODEL", "qwen3.5:4b")
EMBEDDER = "bge-m3"                        # многоязычный эмбеддер, нужен для поиска по памяти
OLLAMA = os.environ.get("OLLAMA_URL", "http://localhost:11434")
CTX = 16384                                # окно на всю лекцию; задаём явно — Ollama молча обрезает лишнее
if IN_COLAB:
    for m in [MODEL, EMBEDDER]:
        !ollama pull {m} > /dev/null 2>&1

import warnings
warnings.filterwarnings("ignore")
import requests, numpy as np, pandas as pd, matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from IPython.display import HTML, display
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from transformers import AutoTokenizer
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

GREEN, ORANGE, RED, BLUE, GREY = "#2a7f62", "#e07a3f", "#c0392b", "#3d6fb6", "#7a8580"
GREEN_SOFT, ORANGE_SOFT, RED_SOFT, BLUE_SOFT = "#d7ece4", "#fbe3d4", "#f6d8d4", "#dbe6f5"
pd.set_option("display.max_colwidth", 120)

def ollama(path, **body):
    r = requests.post(f"{OLLAMA}/api/{path}", json={"model": MODEL, "stream": False, **body}, timeout=3600)
    r.raise_for_status()
    return r.json()

def chat(content, system=None, history=(), **opts):
    # один вызов модели; history — прошлые сообщения [{"role", "content"}]
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": m["role"], "content": m["content"]} for m in history] + [{"role": "user", "content": content}]
    d = ollama("chat", messages=msgs, think=False, options={"temperature": 0, "num_predict": 300, "num_ctx": CTX, **opts})
    chat.last = d                          # сырой ответ: там счётчики токенов и время
    return d["message"]["content"].strip()

def pmap(f, xs, workers=8):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(f, xs))

_emb_cache = {}
def embed(texts):
    # эмбеддинги единичной длины (тогда косинус = скалярное произведение); одинаковые тексты не считаем дважды
    new = [t for t in dict.fromkeys(texts) if t not in _emb_cache]
    for i in range(0, len(new), 64):
        r = requests.post(f"{OLLAMA}/api/embed", json={"model": EMBEDDER, "input": new[i:i + 64]}, timeout=600)
        for t, v in zip(new[i:i + 64], r.json()["embeddings"]):
            _emb_cache[t] = np.array(v) / np.linalg.norm(v)
    return np.array([_emb_cache[t] for t in texts])

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")       # только токенизатор — чтобы честно считать токены
def ntok(text):
    return len(tok.encode(text)) if text else 0

llm = ChatOllama(model=MODEL, base_url=OLLAMA, temperature=0, reasoning=False, num_ctx=CTX, num_predict=1500)
print(chat("Готов к лекции про память? Ответь одним словом.", num_predict=10))

Данные — наш маленький бенчмарк памяти в духе LoCoMo и LongMemEval (о них в конце): **8 пользователей × 8 сессий
переписки** с ассистентом. Как он сделан, важно понимать сразу. Хронологию фактов задал **код**: в какой сессии
пользователь впервые называет свой город, когда меняет работу, заводит собаку, переходит на другую диету. Сами реплики
написали большие модели не из семейства Qwen, а верификатор проверил, что в каждой сессии упомянуты ровно нужные факты.
Поэтому эталон точный, но данные **синтетические**, и местами это видно (ассистент в них бывает рассеян).
Файл вшит в ноутбук, ничего загружать не нужно.

In [ ]:
BENCH_GZ = '''
...
'''
BENCH = json.loads(gzip.decompress(base64.b64decode("".join(BENCH_GZ.split()))))
USERS = BENCH["users"]

def messages_of(sess):
    # «Пользователь: …» / «Ассистент: …» → список сообщений с номером и датой сессии
    blocks = re.split(r"(?m)^(?=(?:Пользователь|Ассистент):)", sess["text"].strip())
    out = []
    for b in blocks:
        m = re.match(r"(Пользователь|Ассистент):\s*(.*)", b.strip(), re.S)
        if m:
            out.append({"role": "user" if m.group(1) == "Пользователь" else "assistant", "content": m.group(2).strip(),
                        "session": sess["session"], "date": sess["date"]})
    for x in out:
        x["ntok"] = ntok(x["content"])
    return out

MSGS = {u["user"]: {s["session"]: messages_of(s) for s in u["sessions"]} for u in USERS}
ALL_MSGS = {name: [m for s in sorted(ss) for m in ss[s]] for name, ss in MSGS.items()}
stats = pd.DataFrame([{"пользователь": u["user"], "сессий": len(u["sessions"]), "сообщений": len(ALL_MSGS[u["user"]]),
                       "токенов истории": sum(m["ntok"] for m in ALL_MSGS[u["user"]]),
                       "вопросов": len(u["questions"])} for u in USERS])
print(BENCH["note"])
stats.set_index("пользователь").T

## Холодное начало: три загадки

### Загадка 1. Стейк для вегетарианца
Сначала самое простое. Модель — это функция: текст на входе, текст на выходе. Скажем ей, как нас зовут, а следующим
вызовом спросим.

In [ ]:
print("1-й вызов:", chat("Привет! Меня зовут Паша, и я вегетарианец.", num_predict=60))
print("2-й вызов:", chat("Как меня зовут и что мне приготовить на ужин? Ответь в одну строку.", num_predict=60))

Ну конечно. Второй вызов ничего не знает о первом: никакого «разговора» у модели нет, есть только текст, который мы
прислали *сейчас*. Чат-приложения создают иллюзию памяти, подклеивая к каждому запросу историю.

Самый популярный способ подклеивать — **буфер**: последние N сообщений (в LangChain это когда-то называлось
`ConversationBufferWindowMemory`). Вот три месяца Паши — восемь сессий. Зелёным — сессии, где он что-то рассказал о себе.

In [ ]:
PASHA = next(u for u in USERS if u["user"] == "Паша")

LABELS = {"city": "город", "job": "работа", "pet": "питомец", "diet": "питание", "hobby": "хобби", "car": "машина"}

def timeline(u):
    cells = []
    for s in u["sessions"]:
        facts = ", ".join(f"{LABELS[k]}: {v}" for k, v in s["facts"].items())
        hot = "вегетарианец" in facts
        bg, bd = (ORANGE_SOFT, ORANGE) if hot else ((GREEN_SOFT, GREEN) if facts else ("#f4f5f4", GREY))
        cells.append(f'<div style="flex:1;background:{bg};border:2px solid {bd};border-radius:8px;padding:6px;'
                     f'font-size:12px;min-height:70px"><b>сессия {s["session"]}</b><br><small>{s["date"]}</small><br>'
                     f'{html.escape(facts) or "—"}</div>')
    display(HTML(f'<div style="display:flex;gap:6px">{"".join(cells)}</div>'))

timeline(PASHA)

Паша стал вегетарианцем в сессии 7, а в сессии 8 рассказывал про новую собаку. Возьмём буфер на 20 последних сообщений
и спросим про ужин.

In [ ]:
DINNER = "Посоветуй, что мне приготовить себе на ужин сегодня? Хочется чего-то сытного. Ответь коротко: одно блюдо и почему."
MEAT = re.compile(r"(мяс\w*|стейк\w*|говядин\w*|куриц\w*|курин\w*|свинин\w*|фарш\w*|бекон\w*|индейк\w*|рыб\w*|лосос\w*|"
                  r"тунц\w*|тунец|креветк\w*|баранин\w*|телятин\w*|шашлык\w*|котлет\w*|колбас\w*|ветчин\w*)", re.I)

def dish_card(title, answer, n_tokens=None):
    body = MEAT.sub(lambda m: f'<b style="color:{RED};background:{RED_SOFT}">{m.group(0)}</b>', html.escape(answer))
    meta = f" · в контексте {n_tokens:,} токенов" if n_tokens is not None else ""
    display(HTML(f'<div style="border-left:4px solid {BLUE};background:#f3f6fb;padding:8px 12px;margin:6px 0;'
                 f'border-radius:4px"><small style="color:{GREY}">{title}{meta}</small><br>{body}</div>'))

buffer20 = ALL_MSGS["Паша"][-20:]
ans_buf = chat(DINNER, history=buffer20)
dish_card("буфер: последние 20 сообщений", ans_buf, chat.last["prompt_eval_count"])
print("в буфере есть фраза о вегетарианстве:", "да" if any("вегетариан" in m["content"].lower() for m in buffer20) else "нет")
print("в ответе мясо или рыба:", "да" if MEAT.search(ans_buf) else "нет")

Смотрим на две строчки под карточкой. Фразы о вегетарианстве в буфере нет: от седьмой сессии в него попал только хвост,
а Паша сказал о мясе в первой трети того разговора. Если в ответе подсвечено мясо — бам! — вот и стейк для вегетарианца.
А если модель предложила что-то овощное, не обольщайтесь: это совпадение, а не память. В восьмой сессии Паша обсуждал
гречку с грибами, и модели было что продолжить.

«Ну так это очевидно, — скажет кто-то. — Не жадничайте, отдайте модели **всю** историю. У Qwen3.5 окно 262 тысячи
токенов, а у Паши за три месяца набралось около десяти тысяч» (смотрите табличку с данными выше).

🗳️ **Ваш прогноз.** Отдаём всю историю, все 8 сессий. Что посоветует модель теперь?

In [ ]:
t0 = time.time()
ans_full = chat(DINNER, history=ALL_MSGS["Паша"])
dish_card("вся история, 8 сессий", ans_full, chat.last["prompt_eval_count"])
print(f"ответ за {time.time() - t0:.1f} с; в ответе мясо или рыба: {'да' if MEAT.search(ans_full) else 'нет'}")

Вот это и есть **загадка 1**. Фраза «мясо и рыбу совсем исключил» лежит прямо в контексте, в седьмой сессии из восьми,
то есть почти в самом конце. Если модель всё равно советует блюдо с мясом, почему? Она же «видит» этот текст!
А если на этот раз обошлось, то не спешите радоваться: один ответ на одну формулировку — это одна монетка. В конце пары
мы зададим четыре разных вопроса про еду, сравним буфер, всю историю и память фактов и посчитаем.

### Загадка 2. Зачем память, если окно — 262 тысячи токенов?
Проверим, что окно правда такое: спросим у самой Ollama, что записано в модели.


In [ ]:
info = ollama("show")["model_info"]
ctx_len = next(v for k, v in info.items() if k.endswith("context_length"))
print(f"{MODEL}: окно контекста {ctx_len:,} токенов — это ~{ctx_len * 3.4 / 1800:.0f} страниц русского текста")
pasha_tok = int(stats.set_index("пользователь").loc["Паша", "токенов истории"])
print(f"вся переписка Паши — {pasha_tok:,} токенов (~{pasha_tok * 3.4 / 1800:.0f} страниц): влезет в окно {ctx_len // pasha_tok} раз")

Вся переписка Паши влезет в это окно много раз подряд: сколько именно, написано в выводе. Если всё влезает, зачем вообще отдельная «память»?
Держите свой ответ в голове: к концу пары у нас будет на него три ответа с цифрами.

### Загадка 3. «Удалите всё, что вы обо мне знаете»
Однажды Паша напишет в поддержку: «Удалите мои данные». По закону о персональных данных (152-ФЗ) он имеет на это право.
Программист открывает базу, удаляет строчку «Паша — вегетарианец» и докладывает: готово.

Готово ли? Сегодня мы построим ассистенту память самых разных видов и в конце проверим, где ещё живёт этот факт.

## 1. Модель без состояния и квадратичный счёт

Итак, модель — чистая функция. Всё, что она «помнит», — это то, что мы сами положили в контекст. Чат-интерфейс на
каждом ходу отправляет **всю историю заново**.

![Каждый ход — заново вся история: n-й запрос длиной ~n, весь диалог ~n²](attachment:stateless.png)

*Каждый ход — заново вся история: n-й запрос длиной ~n, весь диалог ~n²*

Если реплика в среднем стоит $t$ токенов, то на $k$-м ходу мы отправляем $\approx k\,t$ токенов, а за весь диалог из
$n$ ходов:

$$\sum_{k=1}^{n} k\,t = t\,\frac{n(n+1)}{2} \;\sim\; n^2.$$

В лекции 1 мы считали чек за чтение «Мастера и Маргариты»; здесь тот же счётчик, только он тикает квадратично. Посчитаем
на нашем Паше: все его реплики за три месяца. Сравним «каждый раз всю историю» с «последними 2000 токенами».

In [ ]:
def replay_cost(msgs, window=None):
    # сколько входных токенов уходит в модель на каждом ходу пользователя
    per_turn, seen = [], []
    for m in msgs:
        if m["role"] == "user":
            hist = seen if window is None else []
            if window is not None:
                total = 0
                for x in reversed(seen):
                    if total + x["ntok"] > window:
                        break
                    hist.insert(0, x); total += x["ntok"]
            per_turn.append(sum(x["ntok"] for x in hist) + m["ntok"])
        seen.append(m)
    return np.array(per_turn)

full_c, win_c = replay_cost(ALL_MSGS["Паша"]), replay_cost(ALL_MSGS["Паша"], window=2000)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(full_c, color=ORANGE, label="вся история"); ax[0].plot(win_c, color=GREEN, label="окно 2000 токенов")
ax[0].set_xlabel("ход пользователя"); ax[0].set_ylabel("токенов на входе"); ax[0].set_title("Один запрос"); ax[0].legend()
ax[1].plot(full_c.cumsum(), color=ORANGE); ax[1].plot(win_c.cumsum(), color=GREEN)
ax[1].set_xlabel("ход пользователя"); ax[1].set_ylabel("токенов всего"); ax[1].set_title("Весь диалог: парабола против прямой")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()
n, t = len(full_c), full_c[-1] / len(full_c)          # t — средний прирост истории за ход
print(f"{n} реплик Паши: всего {full_c.sum():,} входных токенов против {win_c.sum():,} у окна "
      f"(в {full_c.sum() / win_c.sum():.1f} раза больше)")
print(f"проверка формулой: t·n(n+1)/2 = {t:.0f}·{n}·{n + 1}/2 ≈ {t * n * (n + 1) / 2:,.0f}")

Три месяца переписки одного человека — уже сотни тысяч входных токенов. Умножьте на сто тысяч пользователей.

Есть, правда, хитрость, которая сильно смягчает удар: **кэш префикса**. Если начало запроса совпадает с прошлым,
движку не нужно заново «читать» эти токены: их внутреннее представление (KV-кэш) уже лежит в памяти. Посмотрим, как
это работает в Ollama: два одинаковых длинных запроса подряд.

In [ ]:
nonce = f"[{random.random()}]"                       # уникальное начало — чтобы первый запрос точно был «холодным»
history = [{"role": "user", "content": nonce}] + ALL_MSGS["Паша"]
runs = []
for attempt in ["первый раз", "второй раз"]:
    chat("Как зовут питомца пользователя? Одним словом.", history=history, num_predict=10)
    n_read, sec = chat.last["prompt_eval_count"], chat.last["prompt_eval_duration"] / 1e9
    runs.append((n_read, sec))
    print(f"{attempt}: модель заново прочитала {n_read:,} токенов за {sec:.2f} с")
(n1, s1), (n2, s2) = runs
print(f"длина запроса: {n1:,} токенов (первый вызов — холодный, он читает всё)")
print("вывод:", f"кэш сработал — второй раз прочитано {n2:,} токенов вместо {n1:,}, быстрее в {s1 / max(s2, 1e-3):.0f} раз"
      if n2 < n1 / 2 or s2 < s1 / 3 else "повторного использования нет — оба раза запрос прочитан целиком")

Смотрим на вывод. Если кэш сработал, тот же запрос во второй раз читается в разы быстрее: модель «вспомнила» вычисления,
а не текст. Это тоже память, только вычислительная, и живёт она недолго: пока кэш не вытеснят другие запросы.
Провайдеры API за такие токены берут в разы меньше: подробно про KV-кэш поговорим в лекции 7.

А если повторного использования нет -- не так быстро с выводами «кэш не нужен». Чтобы продолжить с середины, движку
нужно состояние модели ровно на границе префикса. У обычного внимания это просто: KV-кэш — список по токенам, лишнее
отрезали — и готово. Но у Qwen3.5 три четверти слоёв — Gated DeltaNet (раздел 7), и их память — одна матрица, в которую
весь прочитанный текст уже «вмешан». Откатить её на сотню токенов назад нельзя, можно только заранее сохранить снимок.
Умеет ли это конкретная версия движка — вопрос инженерный, и ответ на него напечатан выше.

Но даже работающий кэш экономит деньги, а не внимание модели. Окно в 262 тысячи — это предложение, от которого мы пока не отказались.
Давайте проверим, что будет, если им правда пользоваться.

## 3. Длинный контекст ≠ память

### Цена и задержка
Соберём из переписок всех наших пользователей текст на 8, 32 и 64 тысячи токенов и измерим, сколько модель его
читает. Окно (`num_ctx`) каждый раз выставляем под длину, иначе Ollama молча отрежет начало.

In [ ]:
corpus = "\n\n".join(s["text"] for u in USERS for s in u["sessions"])
corpus_ids = tok.encode(corpus)
print(f"весь корпус переписок: {len(corpus_ids):,} токенов")

rows = []
for size in [8_000, 32_000, 64_000]:
    text = f"[{random.random()}]\n" + tok.decode(corpus_ids[:size - 100])
    d = ollama("generate", prompt=text + "\n\nСколько здесь разных собеседников? Одним числом.", think=False,
               options={"num_ctx": size + 512, "num_predict": 1, "temperature": 0})
    sec = d["prompt_eval_duration"] / 1e9
    rows.append({"токенов": d["prompt_eval_count"], "чтение, с": round(sec, 2), "токенов/с": round(d["prompt_eval_count"] / sec)})
lat = pd.DataFrame(rows)
lat

Сколько памяти уйдёт на такой контекст? У внимания есть **KV-кэш**: для каждого токена в каждом слое хранятся ключ
и значение. Формулу честно выведем в лекции 7, а здесь просто возьмём числа из конфигурации Qwen3.5-4B. Одна
деталь пригодится в разделе 7: полноценное внимание есть не во всех слоях, а только в каждом четвёртом.

In [ ]:
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained("Qwen/Qwen3.5-4B")
cfg = getattr(cfg, "text_config", cfg)
n_full = cfg.layer_types.count("full_attention")
kv_per_token = 2 * n_full * cfg.num_key_value_heads * cfg.head_dim * 2          # K и V, bf16 = 2 байта
print(f"слоёв всего {cfg.num_hidden_layers}, из них с полным вниманием {n_full}; "
      f"KV-кэш: {kv_per_token / 1024:.0f} КБ на токен")
lat["KV-кэш, ГБ"] = (lat["токенов"] * kv_per_token / 1e9).round(2)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(lat["токенов"], lat["чтение, с"], "o-", color=ORANGE, ms=9)
for _, r in lat.iterrows():
    ax.text(r["токенов"], r["чтение, с"], f'  {r["чтение, с"]} с · {r["KV-кэш, ГБ"]} ГБ KV', va="center", fontsize=10)
ax.set_xlabel("токенов в контексте"); ax.set_ylabel("секунд до первого слова"); ax.grid(alpha=.3)
ax.set_xlim(0, lat["токенов"].max() * 1.45); ax.set_ylim(0, lat["чтение, с"].max() * 1.2); ax.set_title("Чем длиннее контекст, тем дольше ждать ответа")
plt.tight_layout(); plt.show()
print(f"полное окно {cfg.max_position_embeddings:,} токенов = {cfg.max_position_embeddings * kv_per_token / 1e9:.1f} ГБ "
      f"KV-кэша на одного пользователя (у T4 16 ГБ, из них доступно около 15 ГБ; веса qwen3.5:4b ~3,4 ГБ)")

Это скорость нашего сервера, где готовился ноутбук; T4 по вычислениям в несколько раз слабее, так что умножайте.
И важно: «чтение» каждого запроса растёт с длиной истории, а ответ пользователь ждёт **до** первого слова.

### Потерянная середина
Хорошо, допустим, мы богаты и терпеливы. Но находит ли модель факт в длинном контексте? Классическая работа
*Lost in the Middle* (Liu et al., 2023) показала: модели лучше всего используют начало и конец контекста, а середину
«проседают». Проверим на нашей модели. Возьмём сессию, где пользователь упоминает факт (например, как зовут питомца),
и спрячем её среди 15 чужих сессий, где этот факт не упоминается: в начало, в середину или в конец.

🗳️ **Ваш прогноз.** Где модель будет ошибаться чаще?

In [ ]:
FIELDS = {   # поле: (вопрос, все возможные значения в бенчмарке)
    "city": ("В каком городе живёт пользователь", ["Казань", "Екатеринбург", "Новосибирск", "Самара", "Пермь", "Тверь",
             "Воронеж", "Калининград", "Томск", "Ярославль", "Москва", "Сочи"]),
    "job": ("Кем работает пользователь", ["инженер-конструктор", "учитель физики", "бариста", "бухгалтер", "сварщик",
            "фронтенд-разработчик", "медсестра", "логист", "архитектор", "аспирант-химик"]),
    "pet": ("Как зовут питомца пользователя (и кто это)", ["кот Барсик", "собака Тоша", "попугай Кеша", "кошка Муся",
            "собака Лайма", "хомяк Пончик", "черепаха Гоша"]),
    "diet": ("Какие ограничения в еде у пользователя", ["вегетарианец", "не ест глютен", "аллергия на орехи",
             "не ест сладкое", "веган", "без ограничений"]),
    "hobby": ("Чем пользователь увлекается в свободное время", ["бегом", "шахматами", "скалолазанием", "керамикой",
              "игрой на гитаре", "фотографией", "велопоходами"]),
    "car": ("На какой машине ездит пользователь", ["Лада Веста", "Kia Rio", "Skoda Octavia", "Haval Jolion",
            "нет машины, ездит на метро"]),
}
KEYS = {   # как значение может прозвучать в ответе (падежи, синонимы); ё заменяем на е заранее
    "Казань": r"казан", "Екатеринбург": r"екатеринбург", "Новосибирск": r"новосиб", "Самара": r"самар",
    "Пермь": r"перм", "Тверь": r"\bтвер[ьи]", "Воронеж": r"воронеж", "Калининград": r"калининград", "Томск": r"томск",
    "Ярославль": r"ярославл", "Москва": r"москв", "Сочи": r"\bсочи\b",
    "инженер-конструктор": r"конструктор", "учитель физики": r"учител", "бариста": r"бариста",
    "бухгалтер": r"бухгалтер", "сварщик": r"сварщик", "фронтенд-разработчик": r"фронтенд|frontend|разработчик",
    "медсестра": r"медсестр|медбрат|медицинск", "логист": r"логист", "архитектор": r"архитектор",
    "аспирант-химик": r"аспирант|химик",
    "кот Барсик": r"барсик", "собака Тоша": r"\bтош", "попугай Кеша": r"\bкеш", "кошка Муся": r"\bмус[яеиюй]",
    "собака Лайма": r"лайм", "хомяк Пончик": r"пончик", "черепаха Гоша": r"\bгош",
    "вегетарианец": r"вегетариан", "не ест глютен": r"глютен", "аллергия на орехи": r"орех",
    "не ест сладкое": r"сладк|сахар|десерт", "веган": r"веган",
    "без ограничений": r"без ограничен|нет ограничен|ограничений нет|никаких ограничен|ест все",
    "бегом": r"\bбег", "шахматами": r"шахмат", "скалолазанием": r"скалолаз", "керамикой": r"керамик|гончар",
    "игрой на гитаре": r"гитар", "фотографией": r"фотограф", "велопоходами": r"велопоход|велосипед",
    "Лада Веста": r"веста|vesta|\bлад[аеуы]\b", "Kia Rio": r"\brio\b|\bрио\b|\bkia\b|\bкиа\b",
    "Skoda Octavia": r"octavia|октави|skoda|шкод", "Haval Jolion": r"jolion|джолион|haval|хавал",
    "нет машины, ездит на метро": r"метро|нет машин|без машин|не водит",
}
FIELD_OF = {v: f for f, (_, vals) in FIELDS.items() for v in vals}
def norm(text):
    return text.lower().replace("ё", "е")
def hit(value, text):
    return re.search(KEYS[value], norm(text)) is not None

In [ ]:
def needle_cases():
    # для каждого пользователя: сессия, где впервые звучит неизменный факт, и сам факт
    cases = []
    for u in USERS:
        changed = {c["field"] for c in u["changes"]}
        for f, s0 in sorted(u["first"].items(), key=lambda x: x[1]):
            sess = u["sessions"][s0 - 1]
            if f not in changed and hit(u["init"][f], sess["text"]):
                cases.append((u, f, sess)); break
    return cases

N_DOCS, POSITIONS = 16, [0, 4, 8, 12, 15]
LONG_CTX = 32768                                      # 16 сессий — это 18–25 тыс. токенов; берём с запасом
rng = random.Random(0)
lim_rows = []
for u, f, needle in needle_cases():
    pool = [s for v in USERS if v is not u for s in v["sessions"]
            if not any(hit(val, s["text"]) for val in FIELDS[f][1])]
    distract = rng.sample(pool, N_DOCS - 1)
    for pos in POSITIONS:
        docs = distract[:pos] + [needle] + distract[pos:]
        text = "\n\n".join(f"### Сессия {i + 1}\n{s['text']}" for i, s in enumerate(docs))
        assert ntok(text) < LONG_CTX - 300, "история не влезает в LONG_CTX — Ollama молча обрезала бы начало"
        ans = chat(FIELDS[f][0] + "? Ответь коротко.", system="Вот история переписки пользователя с ассистентом.\n\n" + text,
                   num_ctx=LONG_CTX, num_predict=30)
        lim_rows.append({"пользователь": u["user"], "поле": f, "позиция": pos, "токенов": ntok(text),
                         "верно": hit(u["init"][f], ans), "ответ": ans})
lim = pd.DataFrame(lim_rows)
print(f"{lim.пользователь.nunique()} фактов × {len(POSITIONS)} позиций, контекст ~{lim.токенов.mean():,.0f} токенов")

In [ ]:
acc = lim.groupby("позиция")["верно"].mean()
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot([p + 1 for p in acc.index], acc.values, "o-", color=BLUE, ms=9)
ax.set_xticks([p + 1 for p in POSITIONS], [f"{p + 1}-я\nиз {N_DOCS}" for p in POSITIONS])
ax.set_ylim(-0.05, 1.05); ax.set_ylabel("доля верных ответов"); ax.grid(alpha=.3)
ax.set_xlabel("где спрятана сессия с фактом"); ax.set_title("Находит ли модель факт в длинной истории")
plt.tight_layout(); plt.show()
edges, middle = acc[[POSITIONS[0], POSITIONS[-1]]].mean(), acc[POSITIONS[1:-1]].mean()
n_case = lim.пользователь.nunique()
print(f"края: {edges:.0%}, середина: {middle:.0%} (на {n_case} фактах одна ошибка — это {1 / n_case:.0%})")
print("вывод:", "середина заметно хуже краёв" if edges - middle > 1.5 / n_case else
      "на прямом вопросе об одном факте провала в середине почти нет — модель нашла иголку")

Посмотрите на вердикт под графиком. Если провала в середине нет, не удивляйтесь: современные модели учили как раз
на «иголках в стоге сена», и прямой вопрос об одном факте они часто вытягивают из любой позиции, в отличие от моделей
2023 года, на которых писали *Lost in the Middle*. Если провал есть — вот она, проседающая середина, даже на прямом
вопросе. Но в обоих случаях вспомните загадку 1: там факт был **почти в конце**, а вопрос — **не прямой** («что
приготовить?»). Найти факт, когда о нём спрашивают, и **применить** факт, когда о нём не спрашивают, — разные умения. Второе для
ассистента важнее: никто не начинает просьбу о рецепте со слов «напоминаю, я вегетарианец».

Итого у длинного контекста три беды: он **дорогой** (квадратичный счёт), **медленный** (секунды до первого слова)
и **рассеивает внимание** (факт есть, но это не значит, что модель его применит). Отсюда и вся идея памяти: класть в контекст не всё,
а **нужное**. Осталось понять, что такое «нужное» и как его найти.

## 4. Как мерить память: бенчмарк и честный судья

Чтобы сравнивать виды памяти, нужен экзамен. У каждого из наших 8 пользователей — 11–14 вопросов пяти типов:

| Тип | Пример | Что проверяет |
| --- | --- | --- |
| current | «В каком городе живёт пользователь сейчас?» | обновление: факт мог смениться |
| past | «На какой машине пользователь ездил раньше?» | история: старое не должно потеряться |
| when | «В какой сессии пользователь сообщил о смене машины?» | время |
| combo | «Чем увлекается и кем работает?» | два факта из разных сессий |
| unknown | «В каком городе живёт пользователь?» — а он не говорил | умение сказать «не знаю» |

Это ровно способности из бенчмарка LongMemEval: извлечение, обновление знаний, рассуждение о времени, несколько
сессий и отказ от ответа.

In [ ]:
qs = pd.DataFrame([dict(q, user=u["user"]) for u in USERS for q in u["questions"]])
print(f"всего вопросов: {len(qs)}")
display(qs.type.value_counts().to_frame("вопросов").T)
qs[qs.user == "Паша"][["type", "q", "answer"]]

Ответы модели будем проверять **кодом**, а не на глаз. Правила простые и строгие:
- current / past — в ответе есть правильное значение (с учётом падежей) **и нет** конкурирующего (старого для current,
  нового для past): «Новосибирск или Самара» — не ответ;
- when — первое число (или порядковое слово) в ответе совпадает с номером сессии;
- combo — оба значения;
- unknown — модель отказалась отвечать и не назвала ни одного значения этого поля.

In [ ]:
REFUSE = re.compile(r"неизвест|не знаю|нет (данных|информац|сведени)|не (упомина|указ|сообща|говорил|называл|известн)|"
                    r"информаци\w* нет|не могу (сказать|ответить)")
ORDINALS = {"перв": 1, "втор": 2, "трет": 3, "четверт": 4, "пят": 5, "шест": 6, "седьм": 7, "восьм": 8}

def first_number(text):
    m = re.search(r"\d+|" + "|".join(ORDINALS), norm(text))
    if not m:
        return None
    return int(m.group(0)) if m.group(0).isdigit() else ORDINALS[m.group(0)]

def judge(q, answer, u):
    a = norm(answer)
    if q["type"] == "unknown":
        return bool(REFUSE.search(a)) and not any(hit(v, a) for v in FIELDS[u["hidden"]][1])
    if REFUSE.search(a) and q["type"] != "when":
        return False
    if q["type"] == "when":
        return first_number(a) == int(q["answer"])
    if q["type"] == "combo":
        return all(hit(v, a) for v in q["answer"].split("; "))
    gold = q["answer"]
    rival = [c["new" if q["type"] == "past" else "old"] for c in u["changes"] if c["field"] == FIELD_OF[gold]]
    return hit(gold, a) and not any(hit(r, a) for r in rival)

for answer in ["Haval Jolion", "раньше Лада Веста, сейчас Haval Jolion", "неизвестно", "на Ладе Весте"]:
    print(f"current «на какой машине ездит сейчас?» ← {answer!r:42} → {judge(PASHA['questions'][4], answer, PASHA)}")

Обратите внимание на вторую строку: «раньше Лада Веста, сейчас Haval Jolion» по сути верно, но код ставит ✗: в ответе
есть конкурирующее значение. Это сознательная строгость: промпт просит ответить одним-двумя словами, и за развёрнутый
ответ модель платит ложным ✗. Лучше так, чем засчитывать «Новосибирск или Самара».

Теперь соберём испытательный стенд. Каждая память умеет две вещи: **записать** сессию (`write`) и **прочитать** то, что
нужно под вопрос (`read`). Прочитанное кладём в системный промпт, вопрос — в сообщение пользователя. Память идёт
**первой**: тогда все вопросы к одному пользователю начинаются одинаково, и кэш префикса из раздела 1, если движок
его поддерживает: сэкономит чтение.

Каждую стратегию будем вносить в **табло**: точность против числа токенов памяти в контексте.

In [ ]:
ANSWER = ("Ты — личный ассистент пользователя. Ниже — твоя память о нём.\n"
          "Ответь на вопрос по памяти коротко: одно-два слова или число, без пояснений. "
          "Если в памяти ответа нет — ответь «неизвестно». Сессии пронумерованы с 1.\n\n=== ПАМЯТЬ ===\n{memory}")

def ask_memory(memory, question):
    return chat(question, system=ANSWER.format(memory=memory or "(пусто)"), num_predict=40)

def render(msgs):
    # сообщения → текст с заголовками сессий
    out, cur = [], None
    for m in msgs:
        if m["session"] != cur:
            cur = m["session"]
            out.append(f"\n### Сессия {cur} ({m['date']})")
        out.append(("Пользователь: " if m["role"] == "user" else "Ассистент: ") + m["content"])
    return "\n".join(out).strip()

class Memory:
    def write(self, msgs): ...                  # сообщения одной сессии
    def read(self, question): return ""         # текст, который пойдёт в контекст

BOARD, MEMS = {}, {}                            # табло: стратегия → ответы; память каждого пользователя

def evaluate(name, make, write_workers=8):
    # make(имя пользователя) → новая память; пишем в неё 8 сессий, потом задаём все вопросы
    def build(u):
        mem = make(u["user"])                      # фабрика получает имя пользователя
        for s in sorted(MSGS[u["user"]]):
            mem.write(MSGS[u["user"]][s])
        return mem
    mems = pmap(build, USERS, write_workers)
    MEMS[name] = dict(zip([u["user"] for u in USERS], mems))
    def run(job):
        u, mem, q = job
        ctx = mem.read(q["q"])
        a = ask_memory(ctx, q["q"])
        return {"стратегия": name, "user": u["user"], "type": q["type"], "q": q["q"], "gold": q["answer"],
                "answer": a, "tokens": ntok(ctx), "ok": judge(q, a, u)}
    df = pd.DataFrame(pmap(run, [(u, m, q) for u, m in zip(USERS, mems) for q in u["questions"]]))
    BOARD[name] = df
    by_type = df.groupby("type")["ok"].mean()
    print(f"{name}: точность {df.ok.mean():.0%}, в среднем {df.tokens.mean():,.0f} токенов памяти | " +
          " ".join(f"{t} {by_type.get(t, 0):.0%}" for t in ["current", "past", "when", "combo", "unknown"]))
    return df

Две крайние точки. **Без памяти** — модель знает только вопрос (это нижняя планка: правильно она ответит разве что
«неизвестно»). **Всё в контекст** — все 8 сессий целиком.

In [ ]:
class NoMemory(Memory):
    pass

class FullHistory(Memory):
    def __init__(self):
        self.msgs = []
    def write(self, msgs):
        self.msgs += msgs
    def read(self, question):
        return render(self.msgs)

_ = evaluate("без памяти", lambda name: NoMemory())
_ = evaluate("всё в контекст", lambda name: FullHistory())

Прежде чем верить цифрам, проверим судью. Правила в коде могут ошибаться: вдруг модель пишет «Тошка» или «пятая сессия».
Для контроля спросим вторую «голову» — ту же модель в роли судьи со структурированным ответом — и посмотрим, где
судьи разошлись. Расхождения прочитаем глазами.

In [ ]:
class Verdict(BaseModel):
    correct: bool

llm_judge = llm.with_structured_output(Verdict, method="json_schema")
JUDGE = ("Эталонный ответ: «{gold}». Ответ ассистента: «{answer}». Вопрос: «{q}». Совпадает ли ответ ассистента "
         "с эталоном по смыслу? Если ассистент называет несколько вариантов или уклоняется — это неверно. "
         "Эталон «неизвестно» означает, что правильно ответить «неизвестно» или «не знаю».")

check = BOARD["всё в контекст"]
check["llm_ok"] = pmap(lambda r: llm_judge.invoke(JUDGE.format(**r)).correct, check.to_dict("records"))
agree = (check.ok == check.llm_ok).mean()
dis = check[check.ok != check.llm_ok]
print(f"судьи согласны в {agree:.0%} случаев; расхождений: {len(dis)} из {len(check)}")
print(f"  код строже (код ✗, LLM-судья ✓): {(~dis.ok).sum()}; LLM-судья строже (код ✓, судья ✗): {dis.ok.sum()}")
if len(dis):
    print("  расхождения по типам вопросов:", dis.groupby("type").size().to_dict())
diff = dis[["type", "q", "gold", "answer", "ok", "llm_ok"]]
diff.rename(columns={"ok": "код", "llm_ok": "LLM-судья"}).style.format(
    {"код": lambda v: "✓" if v else "✗", "LLM-судья": lambda v: "✓" if v else "✗"})

Читаем расхождения глазами — это и есть ручная сверка на подвыборке. Они бывают двух сортов. «Код ✗, судья ✓» — код
строже: в ответе есть и старое значение, или модель написала форму слова, которой нет в наших регулярках. «Код ✓,
судья ✗» — строже судья-4B: отверг верный ответ из-за формулировки или просто ошибся. Сколько каких — напечатано над
таблицей; кто прав в каждой строке, решите сами.

Дальше считаем кодом. Не потому, что он безошибочен, а потому, что он детерминированный, быстрый и **одинаково**
строгий ко всем стратегиям, а его ошибки мы только что видели в лицо. Для сравнения памяти между собой это важнее,
чем абсолютная точность судьи.

Посмотрим, на чём спотыкается «всё в контекст».

In [ ]:
def show_errors(name, n=6):
    df = BOARD[name]
    bad = df[~df.ok].sample(min(n, (~df.ok).sum()), random_state=0)
    rows = "".join(f'<tr><td>{r.type}</td><td>{html.escape(r.q)}</td><td style="color:{GREEN}">{html.escape(r.gold)}</td>'
                   f'<td style="color:{RED}">{html.escape(r.answer[:80])}</td></tr>' for r in bad.itertuples())
    display(HTML(f'<b>{name}: ошибки ({(~df.ok).sum()} из {len(df)})</b><table><tr><th>тип</th><th>вопрос</th>'
                 f'<th>эталон</th><th>ответ</th></tr>{rows}</table>'))

show_errors("всё в контекст")

Вся история под рукой, а ошибки всё равно есть (сколько — в заголовке таблицы). И это самая дорогая точка: каждый
вопрос стоит всей переписки. Можно ли ответить не хуже за малую долю токенов? Начнём с самого дешёвого.

## 5. Краткосрочная память: буфер, окно, сводка

Теперь дешёвые варианты — те, что живут внутри одного контекста.

- **Буфер** — последние $N$ сообщений. Просто, предсказуемо, но то, что выпало за край, потеряно навсегда.
- **Скользящее окно** — то же, но бюджет считаем в токенах: берём сообщения с конца, пока влезают.
- **Сводка** (summary, в агентах её называют compaction) — после каждой сессии модель переписывает краткую сводку:
  $\text{сводка}_s = \text{LLM}(\text{сводка}_{s-1},\ \text{сессия}_s)$. Память почти постоянного размера: мы
  не копим, а сжимаем.

🗳️ **Ваш прогноз.** Какая из трёх лучше ответит на вопрос «на какой машине пользователь ездил *раньше*»?

In [ ]:
class Buffer(FullHistory):
    def __init__(self, n):
        super().__init__(); self.n = n
    def read(self, question):
        return render(self.msgs[-self.n:])

class Window(FullHistory):
    def __init__(self, budget):
        super().__init__(); self.budget = budget
    def read(self, question):
        kept, total = [], 0
        for m in reversed(self.msgs):
            if total + m["ntok"] > self.budget:
                break
            kept.insert(0, m); total += m["ntok"]
        return render(kept)

_ = evaluate("буфер 20 сообщений", lambda name: Buffer(20))
for budget in [1000, 4000]:
    _ = evaluate(f"окно {budget} токенов", lambda name: Window(budget))

In [ ]:
SUMMARIZE = '''Ты ведёшь краткую сводку о пользователе для его ИИ-ассистента.
Перепиши сводку с учётом новой сессии переписки: только устойчивые сведения о самом пользователе
(не его разовые просьбы). Не длиннее 120 слов, без вступлений.

Текущая сводка:
{summary}

Новая сессия:
{session}'''

class Summary(Memory):
    def __init__(self):
        self.text = ""
    def write(self, msgs):
        self.text = chat(SUMMARIZE.format(summary=self.text or "(пока пусто)", session=render(msgs)), num_predict=400)
    def read(self, question):
        return self.text

_ = evaluate("сводка", lambda name: Summary())
pasha_sum = MEMS["сводка"]["Паша"].text
print("\nсводка о Паше после 8 сессий:\n" + pasha_sum + "\n")
for what, pat in [("вегетарианство (сейчас)", r"вегетариан"), ("Лада Веста (раньше)", KEYS["Лада Веста"]),
                  ("номер сессии или дата", r"сесси|\d{4}-\d{2}")]:
    print(f"в сводке есть {what}: {'да' if re.search(pat, norm(pasha_sum)) else 'нет'}")

Читаем сводку Паши и три строчки под ней. Есть ли там нынешняя диета? А машина, на которой он ездил раньше? А хоть
какая-то привязка ко времени? Сводка переписывается целиком, и каждый раз модель решает, что старое уже неважно.
Сжатие с потерями, причём модель сама выбирает, что потерять. Что именно она теряет на всех восьми пользователях,
видно в разбивке по типам вопросов в выводе выше.

Самое время взглянуть на табло.

In [ ]:
def wilson(p, n, z=1.96):
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def show_board(title):
    s = pd.DataFrame({name: {"точность": df.ok.mean(), "токенов": df.tokens.mean(), "n": len(df)}
                      for name, df in BOARD.items()}).T.sort_values("точность")
    lo, hi = np.array([wilson(p, n) for p, n in zip(s["точность"], s["n"])]).T
    fig, ax = plt.subplots(1, 2, figsize=(12, 0.45 * len(s) + 1.4), sharey=True, gridspec_kw={"width_ratios": [3, 2]})
    y = np.arange(len(s))
    ax[0].barh(y, s["точность"], color=GREEN, alpha=.85)
    ax[0].errorbar(s["точность"], y, xerr=[s["точность"] - lo, hi - s["точность"]], fmt="none", ecolor="black", capsize=4)
    for yi, v in zip(y, s["точность"]):
        ax[0].text(0.01, yi, f"{v:.0%}", va="center", color="white", weight="bold")
    ax[0].set_yticks(y, s.index); ax[0].set_xlim(0, 1); ax[0].set_xlabel("точность (усы — 95% интервал)")
    ax[1].barh(y, s["токенов"].clip(lower=1), color=ORANGE, alpha=.85); ax[1].set_xscale("log")
    for yi, v in zip(y, s["токенов"]):
        ax[1].text(max(v, 1) * 1.1, yi, f"{v:,.0f}", va="center", fontsize=9)
    ax[1].set_xlabel("токенов памяти в контексте (лог. шкала)")
    fig.suptitle(title, weight="bold"); plt.tight_layout(); plt.show()

def compare(a, b, types=None, n_boot=4000):
    # парное сравнение на одних и тех же вопросах: разница точности b − a и её 95% бутстреп-интервал
    m = BOARD[a].merge(BOARD[b], on=["user", "q"], suffixes=("_a", "_b"))
    if types:
        m = m[m.type_a.isin(types)]
    d = m.ok_b.astype(float).values - m.ok_a.astype(float).values
    lo, hi = np.percentile(np.random.default_rng(0).choice(d, (n_boot, len(d))).mean(1), [2.5, 97.5])
    verdict = f"«{b}» лучше" if lo > 0 else f"«{a}» лучше" if hi < 0 else "разница в пределах шума"
    print(f"{b} − {a}{' на ' + '/'.join(types) if types else ''}: {d.mean():+.0%} [95%: {lo:+.0%} … {hi:+.0%}], "
          f"{len(d)} вопросов → {verdict}")
    return verdict

def price(base="всё в контекст"):
    # во сколько раз base дороже остальных по токенам памяти
    t = {n: df.tokens.mean() for n, df in BOARD.items() if n != "без памяти"}
    print(f"«{base}» дороже по токенам: " + ", ".join(f"{n} — в {t[base] / max(v, 1):.1f} раза"
                                                    for n, v in t.items() if n != base))

show_board("Табло после раздела 4")
price()
_ = compare("всё в контекст", "окно 4000 токенов")

Читаем по столбцам. Справа — цена: «всё в контекст» стоит в разы дороже остальных (во сколько именно — под
графиком). Слева — качество: дешёвые стратегии теряют то, что выпало за край (буфер и окно) или что модель сочла
неважным (сводка), — насколько это бьёт по точности, видно по столбикам.

Про усы — честно. Это 95% интервал Уилсона, и он оптимистичен: 102 вопроса задают всего 8 пользователей, а вопросы
одного пользователя связаны между собой. Поэтому две стратегии мы сравниваем **парно**, на одних и тех же вопросах:
`compare` печатает разницу точности с бутстреп-интервалом, и вердикт «в пределах шума» выносит код, а не мы.
Оговорка остаётся и тут: бутстреп перемешивает вопросы, а не пользователей, так что и его интервал слегка
оптимистичен — вердикт «лучше» на самой границе считайте поводом проверить на большем наборе, а не открытием.

Это всё была память **в пределах контекста**. Чтобы помнить долго и дёшево, нужна память **снаружи**.

## 6. Долговременная память

### Какой она бывает
Классификацию возьмём у когнитивистов — её перенесли на LLM-агентов в работе **CoALA** (Sumers et al., 2023):
- **семантическая** — факты: «Паша вегетарианец», «у Паши собака Тоша»;
- **эпизодическая** — события с привязкой ко времени: «в сессии 7 Паша просил пасту без мяса»;
- **процедурная** — умения и правила: системный промпт, инструкции, код инструментов.

![CoALA: рабочая память — это контекст; долговременная живёт снаружи и подгружается по запросу](attachment:coala.png)

*CoALA: рабочая память — это контекст; долговременная живёт снаружи и подгружается по запросу*

### Эпизодическая: векторный поиск по репликам
Самое прямолинейное решение — RAG по собственной переписке (подробно про RAG — лекции 4–5). Каждую реплику пользователя
с номером сессии превращаем в эмбеддинг, а на вопрос достаём $k$ самых похожих по косинусу:

$$\text{sim}(q, m) = \frac{\langle e_q, e_m\rangle}{\lVert e_q\rVert\,\lVert e_m\rVert}.$$

Реплики ассистента не индексируем: факты о пользователе говорит пользователь.

In [ ]:
class VectorReplies(Memory):
    def __init__(self, k):
        self.k, self.items, self.vecs = k, [], np.zeros((0, 1024))
    def write(self, msgs):
        new = [f"[сессия {m['session']}, {m['date']}] {m['content']}" for m in msgs if m["role"] == "user"]
        self.items += new
        self.vecs = np.vstack([self.vecs, embed(new)])
    def read(self, question):
        sims = self.vecs @ embed([question])[0]
        top = sorted(np.argsort(-sims)[:self.k])            # найденное — в хронологическом порядке
        return "\n".join(self.items[i] for i in top)

for k in [5, 15]:
    _ = evaluate(f"вектора top-{k}", lambda name: VectorReplies(k))

In [ ]:
q = "На какой машине пользователь ездил раньше?"
mem = MEMS["вектора top-5"]["Паша"]
sims = mem.vecs @ embed([q])[0]
for i in np.argsort(-sims)[:5]:
    print(f"{sims[i]:.3f}  {mem.items[i][:150]}")

Кто-то скажет: «Так это же RAG, задача решена». Но есть нюанс. Поиск находит реплики, *похожие на вопрос*, — а это
не всегда реплики *с ответом*. «На какой машине ездил» похоже
на любой разговор про машины. И даже когда нужная реплика найдена, модели приходится выуживать факт из болтовни.

### Семантическая: извлекаем факты
Идея посильнее: после каждой сессии **извлекать факты** — коротко, отдельными предложениями. Это та же задача
структурного извлечения, что в лекции 2 и ЛР1: JSON Schema + Pydantic, и формат гарантирован. Заодно попросим модель
оценить **важность** факта от 1 до 10 — пригодится при выборке.

In [ ]:
class Fact(BaseModel):
    text: str = Field(description="один факт о пользователе: короткое предложение в третьем лице")
    importance: int = Field(description="важность от 1 до 10 для будущих разговоров: 1 — мелочь, 10 — очень важно")

class Facts(BaseModel):
    facts: list[Fact]

EXTRACT = '''Ты ведёшь память ИИ-ассистента о пользователе. Прочитай сессию переписки и выпиши устойчивые факты
о самом пользователе, которые пригодятся через месяц: где живёт, кем работает, семья и питомцы, питание и здоровье,
увлечения, транспорт, перемены в жизни. НЕ факты: разовые просьбы (рецепт, письмо, маршрут), что лежит
в холодильнике, что пользователь делает сегодня, догадки. Пиши только то, что пользователь сказал прямо.
Если пользователь сообщает, что что-то изменилось, запиши новое состояние и что было раньше.
Важность: 8–10 — город, работа, здоровье и питание, семья, питомцы, машина; 4–7 — увлечения и привычки;
1–3 — мелочи. Если фактов нет — верни пустой список.'''

extractor = ChatPromptTemplate.from_messages([("system", EXTRACT), ("human", "Сессия {s} ({date}):\n\n{session}")]) \
            | llm.with_structured_output(Facts, method="json_schema")

jobs = [(u["user"], s) for u in USERS for s in sorted(MSGS[u["user"]])]
FAILED = []
def extract(job):
    name, s = job
    msgs = MSGS[name][s]
    try:
        return extractor.invoke({"s": s, "date": msgs[0]["date"], "session": render(msgs)}).facts
    except Exception:                  # 4B изредка зацикливается и JSON обрывается на лимите — считаем, что фактов нет
        FAILED.append(job)
        return []
EXTRACTED = dict(zip(jobs, pmap(extract, jobs)))
print(f"извлечено фактов: {sum(len(v) for v in EXTRACTED.values())} из {len(jobs)} сессий; "
      f"не разобрался JSON: {len(FAILED)} сессий")
for s in sorted(MSGS["Паша"]):
    print(f"сессия {s}: " + "; ".join(f"{f.text} ({f.importance})" for f in EXTRACTED[("Паша", s)]))

Первая версия — **просто складывать** все факты в список с номером сессии и весь список отдавать модели.

In [ ]:
class FactsAppend(Memory):
    def __init__(self, name):
        self.name, self.facts = name, []
    def write(self, msgs):
        s = msgs[0]["session"]
        self.facts += [(s, f.text) for f in EXTRACTED[(self.name, s)]]
    def read(self, question):
        return "\n".join(f"- [сессия {s}] {t}" for s, t in self.facts)

_ = evaluate("факты: просто список", FactsAppend)

Дёшево — и неплохо. Но в списке копятся **противоречия**. Возьмём Аню: в сессии 3 она жила в Новосибирске, в сессии 6
переехала в Самару. Что лежит в её памяти про город — и где ассистент будет искать ей мастера?

In [ ]:
ANYA = next(u for u in USERS if u["user"] == "Аня")
anya_mem = MEMS["факты: просто список"]["Аня"]
for s, t in anya_mem.facts:
    if re.search(r"новосиб|самар|переех|город|жив", norm(t)):
        print(f"[сессия {s}] {t}")
REPAIR = "У меня сломалась стиральная машина. В каком городе ты будешь искать мне мастера? Назови только город."
ans_repair = chat(REPAIR, system=ANSWER.format(memory=anya_mem.read(REPAIR)), num_predict=20)
print("\nассистент:", ans_repair)
new_city, old_city = hit("Самара", ans_repair), hit("Новосибирск", ans_repair)
print("вывод:", "Самара — верно" if new_city and not old_city else
      "Новосибирск — мастер поедет по старому адресу" if old_city and not new_city else "ответ неоднозначный")

Посмотрите на вердикт. Даже если модель выбрала Самару (угадать нетрудно, если рядом лежит подсказка вроде «переехала»),
посмотрите на саму память: оба города — равноправные строчки, и угадывать, какая из них свежая, модели приходится при каждом вопросе. А ещё
список только растёт: через год в нём будут сотни строк, и каждая — в каждом запросе.

### Консолидация: ADD, UPDATE, DELETE — как в Mem0
Решение из системы **Mem0** (Chhikara et al., 2025): при записи нового факта найти похожие старые и попросить модель
принять решение:
- **ADD** — новое, добавить;
- **UPDATE** — уточняет или заменяет старый факт;
- **DELETE** — противоречит старому, старый больше не верен;
- **NOOP** — уже известно, ничего не делать.

Похожие старые факты ищем эмбеддингами (top-5 на каждый новый), решение — снова JSON Schema. Заметьте: UPDATE —
единственная операция, которой не было в «просто списке». Именно она и решает спор Новосибирска с Самарой.

![Память фактов: запись с консолидацией и чтение с оценкой](attachment:mem_pipeline.png)

*Память фактов: запись с консолидацией и чтение с оценкой*

In [ ]:
class Op(BaseModel):
    event: Literal["ADD", "UPDATE", "DELETE", "NOOP"]
    id: Optional[int] = Field(None, description="id старого факта для UPDATE и DELETE")
    text: str = Field(description="текст нового или обновлённого факта")
    importance: int = Field(description="важность от 1 до 10")

class Ops(BaseModel):
    ops: list[Op]

CONSOLIDATE = '''Ты обновляешь память ИИ-ассистента о пользователе. Даны старые факты (с id) и новые факты из свежей сессии.
Для каждого нового факта выбери одну операцию:
- ADD — новой информации в памяти нет;
- UPDATE — новый факт заменяет или уточняет старый (укажи id старого и итоговый текст);
- DELETE — старый факт больше не верен (укажи его id);
- NOOP — это уже известно.
Переезд, новая работа, другая машина, другой питомец, другая диета — это UPDATE старого факта, а не ADD.
UPDATE и DELETE — только для старого факта о том же самом (город — с городом, питомец — с питомцем);
факты о разном не трогай.'''

consolidator = ChatPromptTemplate.from_messages([("system", CONSOLIDATE),
                                                 ("human", "Старые факты:\n{old}\n\nНовые факты (сессия {s}):\n{new}")]) \
               | llm.with_structured_output(Ops, method="json_schema")

def consolidate(name):
    # возвращает хранилище: каждый факт с интервалом [since, until) и журнал операций
    store, log = [], []
    def add(text, s, imp):
        store.append({"id": len(store), "text": text, "since": s, "until": None,
                      "importance": min(10, max(1, int(imp)))})     # схема не гарантирует диапазон — обрезаем сами
    for s in sorted(MSGS[name]):
        new = EXTRACTED[(name, s)]
        valid = [f for f in store if f["until"] is None]
        if not new:
            continue
        if not valid:
            for f in new:
                add(f.text, s, f.importance)
            continue
        sims = embed([f.text for f in new]) @ embed([f["text"] for f in valid]).T
        cand = sorted({j for row in sims for j in np.argsort(-row)[:5]})
        try:
            ops = consolidator.invoke({"s": s, "old": "\n".join(f'id={valid[j]["id"]}: {valid[j]["text"]}' for j in cand),
                                       "new": "\n".join(f"- {f.text}" for f in new)}).ops
        except Exception:                                # JSON не разобрался — просто добавляем новые факты
            ops = [Op(event="ADD", text=f.text, importance=f.importance) for f in new]
        for op in ops:
            old = store[op.id] if op.id is not None and 0 <= op.id < len(store) and store[op.id]["until"] is None else None
            if op.event in ("UPDATE", "DELETE") and old is not None:
                old["until"] = s                         # старый факт не стираем, а «закрываем» номером сессии
                log.append((s, op.event, old["text"], op.text if op.event == "UPDATE" else ""))
                if op.event == "UPDATE":
                    add(op.text, s, op.importance)
            elif op.event in ("ADD", "UPDATE"):
                add(op.text, s, op.importance)
                log.append((s, "ADD", "", op.text))
    return store, log

CONSOLIDATED = dict(zip([u["user"] for u in USERS], pmap(consolidate, [u["user"] for u in USERS])))
log = pd.DataFrame(CONSOLIDATED["Аня"][1], columns=["сессия", "операция", "было", "стало"])
log[log.операция != "ADD"]

Теперь у нас есть журнал: что и когда устарело. Из одного и того же хранилища сделаем **две** памяти.

1. **Как в Mem0**: в контекст идут только действующие факты, и без дат: в Mem0 время лежит в метаданных записи,
   а модель видит только текст. Устаревшее как будто стёрто: разве что модель при UPDATE сама вписала старое
   в новый текст («пересел с Лады Весты на Haval»).
2. **Временная**, в духе Zep / Graphiti (Rasmussen et al., 2025): устаревшие факты не стираем, а помечаем интервалом
   действия — «сессии 3–5». Так в графе знаний Graphiti у каждого ребра есть время начала и конца.

In [ ]:
def fmt_fact(f, temporal=True):
    if not temporal:
        return f"- {f['text']}"                     # как в Mem0: только текст, время в промпт не попадает
    if f["until"] is None:
        return f"- [с сессии {f['since']}] {f['text']}"
    return f"- [сессии {f['since']}–{f['until'] - 1}, устарело] {f['text']}"

class FactsStore(Memory):
    def __init__(self, name, temporal):
        self.store, self.temporal = CONSOLIDATED[name][0], temporal
    def facts(self):
        return [f for f in self.store if self.temporal or f["until"] is None]
    def read(self, question):
        return "\n".join(fmt_fact(f, self.temporal) for f in self.facts())

_ = evaluate("факты + консолидация (Mem0)", lambda name: FactsStore(name, temporal=False))
_ = evaluate("факты + время (Graphiti)", lambda name: FactsStore(name, temporal=True))
_ = compare("факты + консолидация (Mem0)", "факты + время (Graphiti)", ["past", "when"])
print("\nпамять Ани о городе (временная версия):")
print("\n".join(fmt_fact(f) for f in CONSOLIDATED["Аня"][0] if re.search(r"новосиб|самар|переех|город", norm(f["text"]))))

### Выборка: давность, важность, релевантность
Пока фактов десятки, их можно отдавать все. Но у живого ассистента через год их будут тысячи. Значит, нужна выборка.
Формулу возьмём у **Generative Agents** (Park et al., 2023) — тех самых двадцати пяти агентов, что жили в виртуальном
городке:

$$\text{score}(m) = \alpha\cdot\text{давность}(m) + \beta\cdot\text{важность}(m) + \gamma\cdot\text{релевантность}(m, q),$$

где давность $= \lambda^{\,s_{\text{now}} - s_m}$ (экспоненциальное забывание), важность — оценка модели при записи,
релевантность — косинус с вопросом; каждое слагаемое нормируем в $[0, 1]$. Берём top-$k$.

У Generative Agents есть и вторая находка — **рефлексия**. Когда суммарная важность свежих воспоминаний переваливает
за порог, агент задаёт себе вопросы о них и записывает выводы более высокого уровня («Паша стал следить за питанием»)
как **новые** воспоминания — со ссылками на исходные. Дальше они участвуют в выборке наравне с остальными. Мы рефлексию
не делаем, но наша сводка из раздела 4 — её близкий родственник.

In [ ]:
class FactsTopK(FactsStore):
    def __init__(self, name, k, w, decay=0.8):
        super().__init__(name, temporal=True)
        self.k, self.w, self.decay = k, np.array(w, float), decay
    def scores(self, question):
        fs = self.facts()
        mm = lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
        now = max(f["since"] for f in fs)
        rec = mm(np.array([self.decay ** (now - f["since"]) for f in fs]))
        imp = mm(np.array([f["importance"] for f in fs], float))
        rel = mm(embed([f["text"] for f in fs]) @ embed([question])[0])
        return fs, np.stack([rec, imp, rel], 1)
    def read(self, question):
        fs, parts = self.scores(question)
        top = sorted(np.argsort(-(parts @ self.w))[:self.k], key=lambda i: fs[i]["since"])
        return "\n".join(fmt_fact(fs[i]) for i in top)

_ = evaluate("факты top-8: релевантность", lambda name: FactsTopK(name, 8, (0, 0, 1)))
_ = evaluate("факты top-8: давность+важность+релевантность", lambda name: FactsTopK(name, 8, (1, 1, 1)))
_ = compare("факты top-8: релевантность", "факты top-8: давность+важность+релевантность")

In [ ]:
mem = MEMS["факты top-8: давность+важность+релевантность"]["Паша"]
fs, parts = mem.scores(DINNER)
order = np.argsort(-parts.sum(1))[:10][::-1]
fig, ax = plt.subplots(figsize=(10, 4.2))
left = np.zeros(len(order))
for j, (lab, c) in enumerate(zip(["давность", "важность", "релевантность"], [GREY, ORANGE, BLUE])):
    ax.barh(range(len(order)), parts[order, j], left=left, color=c, label=lab); left += parts[order, j]
ax.set_yticks(range(len(order)), [fs[i]["text"][:60] for i in order], fontsize=9)
ax.axvline(np.sort(parts.sum(1))[-8], color=RED, ls="--", lw=1, label="порог top-8")
ax.set_xlabel("score"); ax.legend(loc="lower right"); ax.set_title("Что попадёт в контекст на вопрос про ужин")
plt.tight_layout(); plt.show()

Каждый столбик — сумма трёх слагаемых: серое — свежесть, оранжевое — важность, синее — похожесть на вопрос.
Посмотрите, кто наверху. Если там не факты о еде, значит, давность и важность перевесили смысл: важность ставила
та же 4B при записи, и щедрые «восьмёрки» всем подряд делают это слагаемое шумом. Веса $\alpha, \beta, \gamma$ —
гиперпараметры, и подбирать их надо на dev-наборе, как промпт в лекции 2. И ещё: даже у устаревшего факта может быть
высокий score — поэтому мы и храним метку «устарело», чтобы модель не приняла его за действующий.

Обновим табло.

In [ ]:
show_board("Табло после раздела 5")
price()
_ = compare("всё в контекст", "факты + время (Graphiti)")
_ = compare("факты + время (Graphiti)", "факты top-8: давность+важность+релевантность")

Читаем три строчки под табло. Первая — цена: извлечённые факты занимают в контексте в разы меньше всей истории.
Вторая — главный вопрос раздела: теряем ли мы на этом точность, или разница в пределах шума? Третья — что стоит
выборка top-8, которая режет цену ещё сильнее. И вспомните вердикт Mem0 против Graphiti на вопросах «раньше» и «когда»
выше: метки времени должны их возвращать, если только модель не спрятала историю прямо в текст фактов.

Но во всех этих схемах **мы** решали, когда и что записывать: после каждой сессии, отдельным вызовом. А что если
доверить это самой модели?

## 7. Модель сама управляет памятью: MemGPT / Letta

До сих пор память записывалась **в фоне**: сессия закончилась — отдельный вызов извлёк факты и консолидировал их.
Это дёшево и не задерживает ответ, но память отстаёт на целую сессию: скажи Паша «я теперь вегетарианец» и через пять
минут спроси про ужин — фоновая память ещё ничего не знает. Второй режим — запись **прямо в диалоге** (в LangMem его
называют hot path): модель на ходу решает, что запомнить, и следующая же реплика видит обновлённую память.
Классический пример — MemGPT.

**MemGPT** (Packer et al., 2023; сейчас — фреймворк Letta) предлагает смотреть на контекст как на оперативную память
компьютера, а на внешнее хранилище — как на диск. В контексте всегда лежит небольшое **ядро памяти** (core memory) — то,
что о пользователе нужно помнить всегда. Остальное — в архиве. А главное — модель **сама** вызывает функции, чтобы
отредактировать ядро или поискать в архиве. Память становится инструментом, а работа с ней — частью агентного цикла.

![MemGPT: память как в операционной системе](attachment:memgpt.png)

*MemGPT: память как в операционной системе*

Сделаем мини-MemGPT на нашей 4B: две функции для ядра памяти через `bind_tools` из лекции 2. Модель читает реплики
пользователя по одной — как в живом чате, где старые сообщения скоро вытеснят из контекста, — и сама решает, нужно ли
что-то записать.

In [ ]:
from langchain_core.tools import tool

@tool
def core_memory_append(fact: str) -> str:
    "Записать в ядро памяти новый устойчивый факт о пользователе: город, работа, питомцы, питание, увлечения, машина."
    return "ok"

@tool
def core_memory_replace(old: str, new: str) -> str:
    "Заменить в ядре памяти устаревший факт о пользователе новым: переезд, новая работа, другой питомец, другая машина или диета."
    return "ok"

MEMGPT = '''Ты — ассистент с ограниченным контекстом: старые сообщения ты скоро забудешь, и между разговорами
у тебя остаётся только ядро памяти ниже. Если пользователь сообщает о себе устойчивый факт (город, работа, питомцы,
питание и здоровье, увлечения, машина), которого нет в ядре, — вызови core_memory_append. Если факт из ядра устарел —
вызови core_memory_replace. Что лежит в холодильнике и что пользователь делает сегодня — не факты.'''
memgpt = ChatOllama(model=MODEL, base_url=OLLAMA, temperature=0, reasoning=False, num_ctx=CTX, num_predict=150) \
         .bind_tools([core_memory_append, core_memory_replace])

class MemGPTCore(Memory):
    def __init__(self):
        self.core, self.calls = [], []                       # строки ядра: (сессия, текст)
    def apply(self, call, s):
        a = call["args"]
        if call["name"] == "core_memory_replace" and a.get("old") and self.core:
            sims = embed([t for _, t in self.core]) @ embed([a["old"]])[0]
            if sims.max() > 0.6:                          # заменяем самую похожую строку ядра
                self.core.pop(int(np.argmax(sims)))
        text = a.get("new") or a.get("fact")
        if text:
            self.core.append((s, text))
    def write(self, msgs):
        for m in msgs:
            if m["role"] != "user":
                continue
            core = "\n".join(f"- [сессия {s}] {t}" for s, t in self.core) or "(пусто)"
            out = memgpt.invoke([("system", MEMGPT + "\n\nЯдро памяти:\n" + core), ("human", m["content"])])
            self.calls.append((m, [c["name"] for c in out.tool_calls]))
            for c in out.tool_calls:
                self.apply(c, m["session"])
    def read(self, question):
        return "\n".join(f"- [сессия {s}] {t}" for s, t in self.core)

_ = evaluate("MemGPT: модель сама ведёт ядро", lambda name: MemGPTCore())
print("\nядро памяти Паши:\n" + MEMS["MemGPT: модель сама ведёт ядро"]["Паша"].read(""))

Насколько хорошо модель решает, когда писать? У нас есть разметка: в каких сессиях какие факты прозвучали. Реплику
пользователя считаем «с фактом», если в ней есть значение факта этой сессии. Посмотрим, как часто модель вызывала
функцию памяти на репликах с фактом и без.

In [ ]:
rows = []
for u in USERS:
    for m, calls in MEMS["MemGPT: модель сама ведёт ядро"][u["user"]].calls:
        facts = next(s["facts"] for s in u["sessions"] if s["session"] == m["session"])
        rows.append({"с фактом": any(hit(v, m["content"]) for v in facts.values()), "вызвала": bool(calls),
                     "реплика": m["content"]})
mg = pd.DataFrame(rows)
tab = mg.groupby("с фактом")["вызвала"].agg(["mean", "size"])
print(f"реплик с фактом: {tab.loc[True, 'size']}, модель записала в память {tab.loc[True, 'mean']:.0%} из них")
print(f"реплик без факта: {tab.loc[False, 'size']}, записей по ним {tab.loc[False, 'mean']:.0%}")
for title, mask in [("пропущенные факты", mg["с фактом"] & ~mg["вызвала"]),
                    ("записи на репликах без факта", ~mg["с фактом"] & mg["вызвала"])]:
    ex = mg[mask].sample(min(3, mask.sum()), random_state=0)
    print(f"\n{title} — примеры:" if len(ex) else f"\n{title}: нет")
    for t in ex["реплика"]:
        print("  •", t[:120].replace("\n", " "))

Две доли и примеры — это классический компромисс полноты и точности. Пропуск — факт, сказанный вскользь, который
модель не сочла достойным ядра. Запись без факта — это может быть мусор (что пользователь собирается делать
сегодня), а может быть повтор настоящего факта: разметка у нас грубая (реплика считается «с фактом», только если в ней прозвучало новое значение
этой сессии), так что примеры стоит прочитать глазами. Сколько это стоит в точности — на табло. И в любом случае
платим мы вызовом модели на **каждую** реплику, а не на сессию.

### Ещё две архитектуры вкратце
**Временной граф знаний (Zep / Graphiti).** Факты хранятся как рёбра графа «Паша —ездит на→ Haval Jolion», у каждого
ребра — когда факт стал верен и когда перестал. Новый факт, противоречащий старому, не стирает ребро, а закрывает его
датой. Наша «временная» память выше — это тот же приём без графа. Граф добавляет поиск по связям: «кто из друзей Паши
тоже вегетарианец?».

**Файловая память.** Самый простой и, как ни странно, самый популярный в 2025–26 годах подход у агентов для
программирования: агент ведёт заметки в обычных markdown-файлах (`CLAUDE.md`, `AGENTS.md`, `MEMORY.md`), читает их
в начале работы и дописывает по ходу. Ищет по ним не эмбеддингами, а `grep`. Человек может прочитать и поправить
такую память руками — огромный плюс для отладки и для права на удаление. А правила вроде «в этом проекте тесты
запускаются так-то» — это и есть **процедурная** память из CoALA.

In [ ]:
os.makedirs("memory", exist_ok=True)
with open("memory/pasha.md", "w") as f:
    f.write("# Паша\n\n## Действует\n" + "\n".join(fmt_fact(x) for x in CONSOLIDATED["Паша"][0] if x["until"] is None) +
            "\n\n## Устарело\n" + "\n".join(fmt_fact(x) for x in CONSOLIDATED["Паша"][0] if x["until"] is not None) + "\n")
print(open("memory/pasha.md").read()[:1500])
print("$ grep -iE 'веста|haval|машин' memory/pasha.md")
print(subprocess.run(["grep", "-iE", "веста|haval|машин", "memory/pasha.md"], capture_output=True, text=True,
                     env=dict(os.environ, LC_ALL="C.UTF-8")).stdout)        # UTF-8, чтобы -i работал и для кириллицы

Вот она, вся память Паши: текстовый файл, который читает человек и находит `grep`. Бинго — и никаких эмбеддингов.

Заметьте, что общего у всех наших схем: память живёт **снаружи**, а модель каждый раз читает её заново, как в первый
раз. Но ведь внутри модели при чтении длинного текста тоже что-то копится. Это память? И почему она не спасает между
разговорами?

## 8. Память внутри модели

До сих пор память жила **снаружи**. Но и внутри трансформера есть память — KV-кэш, который мы мерили в разделе 2:
он растёт на 32 КБ с каждым токеном. А у Qwen3.5 сюрприз: полноценное внимание только в каждом четвёртом слое.
Остальные три четверти — **Gated DeltaNet** (Yang et al., 2024), разновидность линейного внимания, у которой вместо
растущего кэша — **матрица состояния фиксированного размера**.

![KV-кэш растёт с каждым токеном; состояние DeltaNet — матрица постоянного размера](attachment:deltanet.png)

*KV-кэш растёт с каждым токеном; состояние DeltaNet — матрица постоянного размера*

Как в матрицу $d \times d$ можно «записать» текст? Как в ассоциативную память: пару «ключ → значение» записываем
внешним произведением $v k^\top$, а читаем умножением на ключ: $S k = v (k^\top k) = v$, если $\lVert k\rVert = 1$.
Простое линейное внимание просто копит: $S_t = S_{t-1} + v_t k_t^\top$. Беда в том, что ключи не ортогональны, и записи
мешают друг другу. **Дельта-правило** сначала читает, что уже лежит под ключом, и записывает только поправку:

$$S_t = S_{t-1} + \beta_t\,(v_t - S_{t-1}k_t)\,k_t^\top = S_{t-1}(I - \beta_t k_t k_t^\top) + \beta_t v_t k_t^\top,$$

а **гейт** $\alpha_t \in (0, 1)$ умножает всё старое состояние — это управляемое забывание.

Узнаёте? Это тот же сюжет, что с Mem0 в разделе 5, только внутри слоя нейросети. Простое накопление — это **ADD**:
после смены факта под ключом лежит каша из старого и нового значения. Дельта-правило читает старое и заменяет его —
это **UPDATE**. А гейт $\alpha$ — забывание: он разом приглушает всё старое, когда модель решает, что пора.
Цена фиксированного размера тоже знакомая: в матрицу $d \times d$ не положишь больше $d$ независимых пар, и когда
записей больше, деградирует любая такая память. Обе памяти мы напишем с нуля и устроим им экзамен в лекции 10 —
сегодня нам важен вывод: выбор между «буфером» и «сводкой» модель делает и у себя внутри.

Сравним настоящие размеры для Qwen3.5-4B.

In [ ]:
n_lin = cfg.layer_types.count("linear_attention")
state = n_lin * cfg.linear_num_value_heads * cfg.linear_key_head_dim * cfg.linear_value_head_dim * 4   # fp32
L = np.array([1e3, 8e3, 32e3, 128e3, 262e3])
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(L, L * kv_per_token / 1e9, "o-", color=ORANGE, label=f"KV-кэш {n_full} слоёв с полным вниманием")
ax.plot(L, L * kv_per_token * cfg.num_hidden_layers / n_full / 1e9, "o--", color=RED,
        label=f"если бы внимание было во всех {cfg.num_hidden_layers} слоях")
ax.plot(L, np.full(len(L), state / 1e9), "o-", color=GREEN, label=f"состояние {n_lin} слоёв DeltaNet")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("токенов в контексте"); ax.set_ylabel("ГБ")
ax.legend(); ax.grid(alpha=.3); ax.set_title("Память Qwen3.5-4B на одного пользователя")
plt.tight_layout(); plt.show()
print(f"состояние DeltaNet: {state / 1e6:.0f} МБ при любой длине; KV-кэш на 262k токенов: "
      f"{262_144 * kv_per_token / 1e9:.1f} ГБ (а без гибрида было бы "
      f"{262_144 * kv_per_token * cfg.num_hidden_layers / n_full / 1e9:.0f} ГБ)")

Гибрид — компромисс: DeltaNet дёшево несёт «общее впечатление» обо всём тексте, а редкие слои полного внимания умеют
точно найти конкретное место. Подробно про внимание и линейные слои — в лекции 10.

Куда это движется дальше (обзорно):
- **сжатие контекста в специальные токены** — модель учат сворачивать длинный промпт в несколько «gist-токенов»
  (Mu et al., 2023);
- **Titans** (Behrouz et al., 2025, Google) — память-нейросеть, которая **дообучается во время инференса**: чем сильнее
  новый токен «удивил» модель (больше градиент ошибки), тем сильнее он записывается; плюс забывание, как наш гейт $\alpha$.

Вся эта память живёт ровно один запрос. Между разговорами её нет — поэтому внешняя память ассистенту нужна всё равно.

## 9. Мультимодальная память: видео, которое не влезает

Ассистент с камерой (умные очки, робот, видеорегистратор) получает не реплики, а **поток кадров**. Посчитаем:
7-минутное видео при 25 кадрах в секунду — это около десяти тысяч кадров (точное число напечатаем ниже). Даже если
кадр маленький и стоит VLM порядка сотни токенов, семь минут — это уже около миллиона токенов, а час — вообще без шансов.

Решение то же, что для диалогов: **память вместо всех кадров в контексте**. Идея общая у целого семейства работ,
а устроены они по-разному:
- **MovieChat** (Song et al., 2024) держит внутри модели кратковременную память из токенов недавних кадров и сливает
  похожие соседние токены в компактную долговременную;
- **MA-LMM** (He et al., 2024) читает видео онлайн, кадр за кадром, и копит банк визуальных признаков (и запросов
  Q-Former), сжимая похожие соседние записи;
- два разных **VideoAgent** держат **текстовую** память снаружи модели: у Wang et al. (2024) LLM-агент читает подписи
  и итеративно решает, какие кадры досмотреть (их находит CLIP), у Fan et al. (2024) — временная память подписей
  к отрезкам видео и память объектов, по которым агент ищет инструментами.

Мы сделаем вариант, ближе всего к VideoAgent: кадр раз в $N$ секунд → VLM пишет подпись → храним «время + подпись +
эмбеддинг» → на вопрос ищем по памяти. Эпизодическая память в чистом виде.

![Видео → эпизодическая память кадров → поиск по вопросу](attachment:video_memory.png)

*Видео → эпизодическая память кадров → поиск по вопросу*

Наш ролик — [«Baking bread at home»](https://commons.wikimedia.org/wiki/File:Baking_bread_at_home.ogv) с Wikimedia
Commons (автор Arquus, конвертация в OGV — Leonel Sohns, лицензия
[CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)): семь минут, как человек печёт хлеб. Кадры достанем
`ffmpeg` раз в 10 секунд и увеличим вдвое: ролик в 240p, а граммы и градусы написаны на плашках мелким шрифтом —
пусть на надпись придётся больше патчей. Подписи сделает **та же** qwen3.5:4b — она умеет смотреть на картинки
(подробно про VLM — лекция 11).

In [ ]:
VIDEO_URL = "https://upload.wikimedia.org/wikipedia/commons/4/48/Baking_bread_at_home.ogv"
VIDEO, STEP = "data/bread.ogv", 10
os.makedirs("data/frames", exist_ok=True)
if not os.path.exists(VIDEO):
    r = requests.get(VIDEO_URL, headers={"User-Agent": "HSE-LLM-course/1.0 (education)"}, timeout=600)
    r.raise_for_status(); open(VIDEO, "wb").write(r.content)
probe = json.loads(subprocess.run(["ffprobe", "-v", "error", "-print_format", "json", "-show_format", "-show_streams", VIDEO],
                                  capture_output=True, text=True).stdout)
vs = next(s for s in probe["streams"] if s["codec_type"] == "video")
duration, fps = float(probe["format"]["duration"]), eval(vs["r_frame_rate"])
for f in os.listdir("data/frames"):
    os.remove(f"data/frames/{f}")
# кадр в момент 0, STEP, 2·STEP, … секунд (select по времени кадра, а не fps-фильтр — так таймкоды точные), ×2 по размеру
subprocess.run(["ffmpeg", "-v", "error", "-i", VIDEO, "-vf", f"select='isnan(prev_selected_t)+gte(t-prev_selected_t,{STEP - 0.02})',scale=iw*2:ih*2",
                "-vsync", "vfr", "-q:v", "3", "data/frames/f%03d.jpg"], check=True)
FRAMES = [(i * STEP, f"data/frames/{f}") for i, f in enumerate(sorted(os.listdir("data/frames")))]
print(f"{duration:.0f} с, {vs['width']}×{vs['height']}, {fps:.0f} кадров/с → всего {duration * fps:,.0f} кадров; "
      f"берём {len(FRAMES)}")
mmss = lambda t: f"{int(t) // 60:02d}:{int(t) % 60:02d}"
fig, axes = plt.subplots(2, 6, figsize=(15, 4.4))
for ax, (t, path) in zip(axes.flat, FRAMES[::max(1, len(FRAMES) // 12)]):
    ax.imshow(plt.imread(path)); ax.set_title(mmss(t), fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

Вопросы к видео составили мы сами, посмотрев ролик; эталоны проверяются кодом. Есть вопросы «что», «сколько»,
«когда» (ответ — минута и секунда; засчитываем попадание в интервал, где это видно на экране), один на сложение
нескольких фактов и один с подвохом.

In [ ]:
def to_sec(answer):
    m = re.search(r"(\d{1,2})[:.](\d{2})", answer)
    if m:
        return int(m.group(1)) * 60 + int(m.group(2))
    m = re.search(r"(\d+)\s*мин\w*(?:\s*(\d+)\s*сек)?", norm(answer))
    return int(m.group(1)) * 60 + int(m.group(2) or 0) if m else None

def when_in(lo, hi):
    return lambda a: (lambda s: s is not None and lo <= s <= hi)(to_sec(a))

has = lambda a, p: re.search(p, norm(a)) is not None
VQ = [
    ("Что стояло на кухонных весах в начале видео?", lambda a: has(a, r"стакан|кувшин|емкост|чаш|мерн|миск")),
    ("Что добавили в мерный стакан вместе с солью?", lambda a: has(a, r"\bмед")),
    ("Какие специи добавили в хлеб?", lambda a: sum(has(a, p) for p in [r"фенхел", r"кориандр", r"анис"]) >= 2),
    ("Сколько граммов ржаной муки взяли?", lambda a: "600" in a),
    ("Сколько граммов полбяной муки взяли?", lambda a: "400" in a),
    ("Сколько времени тесто подходило в первый раз?", lambda a: has(a, r"(?<!пол)час|60 мин")),
    ("До какой температуры разогрели духовку?", lambda a: "230" in a),
    ("Что сделали с хлебом ножом перед выпечкой?", lambda a: has(a, r"надрез|разрез|насеч|полос|лини|порез")),
    ("Сколько минут в сумме хлеб пёкся в духовке?", lambda a: has(a, r"\b60\b|(?<!пол)час")),
    ("На какой минуте видео тесто начинают месить руками? Ответь в формате ММ:СС.", when_in(235, 330)),
    ("На какой минуте видео хлеб ставят в духовку? Ответь в формате ММ:СС.", when_in(340, 370)),
    ("Есть ли в видео собака?", lambda a: has(a, r"\bнет\b|не видно|не появляется|отсутств")),
]
print(f"{len(VQ)} вопросов")

In [ ]:
def b64(path):
    return base64.b64encode(open(path, "rb").read()).decode()

def vlm(prompt, paths=(), **opts):
    d = ollama("chat", messages=[{"role": "user", "content": prompt, "images": [b64(p) for p in paths]}], think=False,
               options={"temperature": 0, "num_predict": 120, "num_ctx": CTX, **opts})
    return d["message"]["content"].strip(), d["prompt_eval_count"]

CAPTION = ("Это кадр из видео. Опиши его одним-двумя предложениями по-русски: что видно и что делают. "
           "Если на кадре есть надпись — перепиши её дословно.")
t0 = time.time()
caps = pmap(lambda fr: vlm(CAPTION, [fr[1]]), FRAMES)
VMEM = [{"t": t, "caption": c} for (t, _), (c, _) in zip(FRAMES, caps)]
tok_per_frame = np.mean([n for _, n in caps]) - ntok(CAPTION)
vecs = embed([m["caption"] for m in VMEM])
print(f"{len(VMEM)} подписей за {time.time() - t0:.0f} с; кадр стоит ~{tok_per_frame:.0f} токенов на входе VLM")
print("числа с плашек попали в подписи:", {w: any(w in m["caption"] for m in VMEM) for w in ["600", "400", "230"]})
for m in VMEM[::6]:
    print(f"[{mmss(m['t'])}] {m['caption'][:160]}")

Теперь пять способов ответить на вопросы:
1. **один кадр** из середины видео — «VLM без памяти»;
2. **8 кадров разом** — равномерно по видео, все картинки в один запрос;
3. **все подписи в контекст** — текстовая эпизодическая память целиком;
4. **поиск** — top-5 подписей по эмбеддингу вопроса, с таймкодами;
5. **поиск + пересмотр** — находим top-3 подписи и показываем модели сами эти кадры (так делает VideoAgent: память
   говорит, *куда* смотреть).

🗳️ **Ваш прогноз.** Кто ответит на «когда хлеб поставили в духовку?»

In [ ]:
VSYS = "Отвечай коротко по-русски, одним предложением."
mid = FRAMES[len(FRAMES) // 2]
uni = [FRAMES[int(i)] for i in np.linspace(0, len(FRAMES) - 1, 8)]
mem_text = lambda ms: "\n".join(f"[{mmss(m['t'])}] {m['caption']}" for m in ms)

def v_one(q):
    return vlm(f"{VSYS} Это кадр из видео ({mmss(mid[0])}).\n\n{q}", [mid[1]])
def v_eight(q):
    stamps = ", ".join(f"кадр {i + 1} — {mmss(t)}" for i, (t, _) in enumerate(uni))
    return vlm(f"{VSYS} Это 8 кадров из видео по порядку ({stamps}).\n\n{q}", [p for _, p in uni])
def v_all(q):
    return vlm(f"{VSYS} Вот подписи к кадрам видео с таймкодами:\n{mem_text(VMEM)}\n\n{q}")
def v_search(q, k=5):
    top = sorted(np.argsort(-(vecs @ embed([q])[0]))[:k])
    return vlm(f"{VSYS} Вот найденные в памяти моменты видео:\n{mem_text([VMEM[i] for i in top])}\n\n{q}")
def v_look(q, k=3):
    top = sorted(np.argsort(-(vecs @ embed([q])[0]))[:k])
    stamps = ", ".join(f"кадр {j + 1} — {mmss(VMEM[i]['t'])}" for j, i in enumerate(top))
    return vlm(f"{VSYS} Вот кадры из видео, найденные по подписям ({stamps}). Подписи:\n"
               f"{mem_text([VMEM[i] for i in top])}\n\n{q}", [FRAMES[i][1] for i in top])

VSTRAT = {"один кадр": v_one, "8 кадров разом": v_eight, "все подписи": v_all, "поиск top-5": v_search,
          "поиск + пересмотр": v_look}
vres = {name: pmap(lambda qa: f(qa[0]), VQ) for name, f in VSTRAT.items()}
vtab = pd.DataFrame({name: [check(a) for (q, check), (a, _) in zip(VQ, res)] for name, res in vres.items()},
                    index=[q.split(" Ответь")[0] for q, _ in VQ])
vtok = {name: np.mean([n for _, n in res]) for name, res in vres.items()}
print("токенов на вопрос:", {k: f"{v:,.0f}" for k, v in vtok.items()})
print("точность:", {k: f"{v:.0%}" for k, v in vtab.mean().items()})
vtab.style.format(lambda v: "✓" if v else "✗") \
    .map(lambda v: f"background-color:{GREEN_SOFT if v else RED_SOFT};color:{GREEN if v else RED}")

In [ ]:
i_oven = next(i for i, (q, _) in enumerate(VQ) if "духовку? Ответь" in q)
print(VQ[i_oven][0])
for name in VSTRAT:
    print(f"{name:>18}: {vres[name][i_oven][0][:110]}")
hour = 3600 / STEP * tok_per_frame
print(f"\nчас видео, кадр раз в {STEP} с: {hour:,.0f} токенов картинками; раз в секунду: {3600 * tok_per_frame:,.0f} — "
      f"окно {ctx_len:,}")

Сверим табличку с тем, чего стоит ждать от каждой стратегии. Один кадр не знает ни прошлого, ни будущего: на «когда»
и «сколько в сумме» ему остаётся гадать. Восемь кадров разом видят больше, но между ними почти по минуте пропусков,
а каждый кадр — это ещё порция токенов (сколько — в строке «токенов на вопрос»). Текстовая память подписей дешёвая
и знает **время**: таймкод записан рядом с подписью, как номер сессии рядом с фактом. Её главный риск — подпись теряет
детали, которые VLM не сочла важными или не смогла прочитать: строчка «числа с плашек попали в подписи» показывает,
дошли ли до памяти граммы и градусы. Тут выручает «пересмотр»: по памяти находим момент, а потом смотрим на сам кадр.
И помните про масштаб: вопросов 12, одна ошибка — это 8 п.п., так что разница в один-два ответа — шум.

А теперь представьте очки с камерой и микрофоном, которые носят весь день: речь превращается в текст (Whisper,
лекция 11), кадры — в подписи, и всё это ложится в ту же эпизодическую память с таймкодами. «Где я оставил ключи?»
— это поиск по памяти, а не по видео. И, разумеется, это самые чувствительные персональные данные, какие только
бывают. Что возвращает нас к загадке 3.

## 10. Право на забвение

Память — это персонализация, а персонализация — это деньги: CRM-ассистенты, поддержка, которая «помнит» клиента.
Но факты о пользователе — **персональные данные**. По 152-ФЗ «О персональных данных» человек вправе потребовать их
уточнения, блокирования или уничтожения, а для данных о здоровье (диета, аллергия — уже близко) требования ещё строже.
Подробно про безопасность — в лекции 13.

Паша пишет: «Удалите всё, что вы знаете о моём питании». Программист удаляет из базы фактов строчки про вегетарианство.
Посмотрим, где ещё живёт этот факт.

In [ ]:
SECRET = r"вегетариан|мяс\w* и рыб|без мяса"
def where(name):
    m = MEMS[name]["Паша"]
    if hasattr(m, "msgs"):
        return [x["content"] for x in m.msgs]
    if hasattr(m, "items"):
        return m.items
    if hasattr(m, "text"):
        return [m.text]
    if hasattr(m, "store"):
        return [f["text"] for f in m.store]
    if hasattr(m, "core"):
        return [t for _, t in m.core]
    if isinstance(getattr(m, "facts", None), list):
        return [t for _, t in m.facts]
    return []

PLACES = {"история (буфер, окно)": "всё в контекст", "сводка": "сводка", "векторная база": "вектора top-5",
          "факты: просто список": "факты: просто список", "факты с консолидацией": "факты + время (Graphiti)",
          "ядро MemGPT": "MemGPT: модель сама ведёт ядро"}
def audit():
    out = {place: sum(bool(re.search(SECRET, norm(t))) for t in where(name)) for place, name in PLACES.items()}
    out["кэш извлечений"] = sum(bool(re.search(SECRET, norm(f.text)))
                                for (who, _), fs in EXTRACTED.items() if who == "Паша" for f in fs)
    return out

before = audit()
BACKUP = copy.deepcopy(CONSOLIDATED["Паша"])          # копия — чтобы в разгадке вернуть память как была
# «удалили из базы»: только факты
store = CONSOLIDATED["Паша"][0]
store[:] = [f for f in store if not re.search(SECRET, norm(f["text"]))]
after_db = audit()
pd.DataFrame({"до удаления": before, "удалили из базы фактов": after_db}).style \
    .map(lambda v: f"background-color:{RED_SOFT if v else GREEN_SOFT}") \
    .format(lambda v: f"{v} записей" if v else "чисто")

Факт живёт везде, куда мы его записали: в истории сообщений, в сводке, в векторной базе — смотрите таблицу. Отдельно
взгляните на строки «факты: просто список» и «кэш извлечений»: это промежуточные данные, о которых программист,
удалявший «строчку из базы», даже не вспомнил. Удалить надо из всех мест, и у каждого — свои сложности. Сообщения и вектора можно найти и вычеркнуть. Сводку «вычеркнуть» нельзя:
факт растворён в тексте, её придётся переписать. Почистим всё — и спросим ассистента, что он помнит о питании Паши.

In [ ]:
for mems in MEMS.values():                            # все истории сообщений: всё в контекст, буфер, окна
    m = mems["Паша"]
    if hasattr(m, "msgs"):
        m.msgs = [x for x in m.msgs if not re.search(SECRET, norm(x["content"]))]
fl = MEMS["факты: просто список"]["Паша"]
fl.facts = [(s, t) for s, t in fl.facts if not re.search(SECRET, norm(t))]
vm = MEMS["вектора top-5"]["Паша"]
keep = [i for i, t in enumerate(vm.items) if not re.search(SECRET, norm(t))]
vm.items, vm.vecs = [vm.items[i] for i in keep], vm.vecs[keep]
core = MEMS["MemGPT: модель сама ведёт ядро"]["Паша"]
core.core = [(s, t) for s, t in core.core if not re.search(SECRET, norm(t))]
sm = MEMS["сводка"]["Паша"]
sm.text = chat("Перепиши сводку, полностью убрав всё о питании и пищевых ограничениях пользователя. "
               "Остальное оставь как есть.\n\n" + sm.text, num_predict=400)
after_all = audit()
DIET_Q = "Какие ограничения в еде у пользователя сейчас?"
leak = {place: ask_memory(MEMS[name]["Паша"].read(DIET_Q), DIET_Q) for place, name in PLACES.items()}
pd.DataFrame({"после чистки": {k: f"{v} записей" if v else "чисто" for k, v in after_all.items()},
              "ассистент отвечает о питании": {k: leak.get(k, "—") for k in after_all}})

Кэш извлечений мы нарочно не трогали. Если в нём остались записи (смотрите таблицу), то это сырьё, из которого
собирается память фактов: стоит завтра пересобрать память (новый промпт консолидации, миграция базы) — и удалённый
факт **воскреснет**. Производные данные удаляются вместе с источником.

Даже после вычёркивания по ключевым словам в истории могут остаться косвенные следы: ответ ассистента «паста без …»,
рецепт из одних овощей. Модель — отличный детектив и иногда восстанавливает удалённое по уликам. Чтобы удалить
по-настоящему, нужно удалять **сессии целиком** (или все записи, порождённые из них), помнить про резервные копии,
логи запросов, кэши — и про данные, на которых модель **дообучали**: из весов «вычеркнуть» нельзя (лекция 12).

Практический вывод для архитектуры: храните у каждой записи памяти **ссылку на источник** (сессию, сообщение). Тогда
«удалить всё, что выведено из сессии 7» — это запрос к базе, а не археология.

## 11. Как это мерят "взрослые"

Наш бенчмарк — игрушечная версия двух настоящих:
- **LoCoMo** (Maharana et al., 2024) — очень длинные диалоги двух собеседников на десятки сессий; вопросы на один факт,
  на несколько сессий, на время, на «открытый мир» и с подвохом;
- **LongMemEval** (Wu et al., 2024) — 500 вопросов к истории чата с ассистентом; пять способностей: извлечение,
  рассуждение по нескольким сессиям, по времени, обновление знаний и отказ от ответа.

Именно на них сравнивают Mem0, Zep, Letta и другие системы памяти. Цифры в статьях разных систем часто несравнимы
(разные модели, судьи и настройки) — поэтому мы и собрали стенд, где всё одинаково, кроме памяти.

## 🏆 Табло

In [ ]:
summary = pd.DataFrame({name: {"точность": df.ok.mean(), "токенов": df.tokens.mean()} for name, df in BOARD.items()}).T
def color_of(name):
    for key, c in [("без", GREY), ("всё", ORANGE), ("буфер", BLUE), ("окно", BLUE), ("сводка", BLUE),
                   ("вектора", RED), ("MemGPT", RED), ("факты", GREEN)]:
        if name.startswith(key):
            return c
    return GREY
markers = "osD^vP*Xhp<>8d"
fig, ax = plt.subplots(figsize=(11, 5.5))
for (name, r), mk in zip(summary.iterrows(), markers * 2):
    c = color_of(name)
    ax.scatter(max(r["токенов"], 1), r["точность"], s=140, marker=mk, color=c, edgecolor="black", zorder=3, label=name)
front = summary.sort_values("токенов")
front = front[front["точность"] >= front["точность"].cummax()]
ax.plot(front["токенов"].clip(lower=1), front["точность"], "--", color=GREY, zorder=1, label="граница Парето")
ax.set_xscale("log"); ax.set_xlabel("токенов памяти в контексте на вопрос (лог. шкала)"); ax.set_ylabel("точность")
ax.set_ylim(-0.02, 1.02); ax.grid(alpha=.3); ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9)
ax.set_title("Точность против цены: кто на границе Парето"); plt.tight_layout(); plt.show()
print("на границе Парето:", ", ".join(front.index))

In [ ]:
types = ["current", "past", "when", "combo", "unknown"]
heat = pd.DataFrame({name: df.groupby("type")["ok"].mean() for name, df in BOARD.items()}).T[types]
heat = heat.loc[summary.sort_values("точность").index]
fig, ax = plt.subplots(figsize=(8.5, 0.42 * len(heat) + 1.2))
ax.imshow(heat.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat.values[i, j]:.0%}", ha="center", va="center", fontsize=9)
counts = qs.type.value_counts()
ax.set_xticks(range(len(types)), [f"{t}\n({counts[t]} вопр.)" for t in types]); ax.set_yticks(range(len(heat)), heat.index)
ax.set_title("Кто на чём ломается"); plt.tight_layout(); plt.show()
for t in ["past", "when", "unknown"]:
    col = heat[t].drop("без памяти", errors="ignore").sort_values()
    print(f"{t:>7}: хуже всех — {', '.join(col.index[:3])}; лучше всех — {', '.join(col.index[::-1][:3])}")
_ = compare("факты + консолидация (Mem0)", "факты + время (Graphiti)", ["past", "when"])

Читаем тепловую карту по столбцам — у каждой памяти свой профиль провалов. По устройству памяти ожидаем вот что,
а сверяем с выводом под картой:
- **past** и **when** должны проваливать те, кто стирает историю: буфер и окно (забыли начало), сводка (переписала),
  консолидация в стиле Mem0 (UPDATE закрыл старое, а дат в промпте нет). Спасает время рядом с фактом — номер сессии,
  интервал действия. Если Mem0 держится на past лучше, чем мы предсказали, загляните в его факты: модель при UPDATE
  нередко сама пишет «пересел с Лады на Haval», и история прячется в тексте;
- **current** страдает там, где старое и новое лежат рядом без пометок;
- **unknown** — индикатор фантазии: чем больше лишнего текста в контексте, тем охотнее модель «вспоминает» то, чего
  не было (в combo-вопросах столбик короткий — смотрите на число вопросов).

## Разгадки

**Загадка 1 — стейк для вегетарианца.** Буфер фразы о вегетарианстве просто не видел — мы проверили это кодом. Вся
история видела. Если она всё же посоветовала мясо, то вот почему: в ней шесть сессий с курицей против одной фразы,
и вопрос «что приготовить» не просит модель ничего вспоминать. Длинный контекст хранит факт, но не обязательно
делает его **заметным**. Память фактов кладёт в контекст коротко и прямо: «вегетарианец, с сессии 7». Проверим не на одном вопросе, а на четырёх — одинаковых для всех трёх вариантов.
Вегетарианское ли блюдо, решает модель-судья со структурированным ответом, а для перекрёстной проверки — наш словарь
мяса `MEAT` из загадки.

In [ ]:
DINNERS = [DINNER, "В субботу идём с друзьями в ресторан. Что мне там заказать? Предложи одно конкретное блюдо.",
           "Хочу пожарить что-нибудь на гриле на даче в выходные. Что посоветуешь? Одно блюдо.",
           "Что взять с собой на обед в офис завтра? Одно блюдо."]
class Veg(BaseModel):
    vegetarian: bool = Field(description="true, если в предложенном блюде нет мяса, птицы, рыбы и морепродуктов")
veg_judge = llm.with_structured_output(Veg, method="json_schema")
CONSOLIDATED["Паша"] = BACKUP                               # в разделе 9 мы почистили память — вернём копию
facts_mem = FactsStore("Паша", temporal=True)
VARIANTS = {"буфер 20 сообщений": lambda q: dict(history=ALL_MSGS["Паша"][-20:]),
            "вся история": lambda q: dict(history=ALL_MSGS["Паша"]),
            "память фактов": lambda q: dict(system="Что ты помнишь о пользователе:\n" + facts_mem.read(q))}
print(f"память фактов о Паше: {ntok(facts_mem.read(''))} токенов против {ntok(render(ALL_MSGS['Паша'])):,} у всей истории")
rows = []
for name, ctx in VARIANTS.items():
    for q in DINNERS:
        a = chat(q, num_predict=150, **ctx(q))
        rows.append({"память": name, "вопрос": q[:40] + "…", "ответ": a,
                     "вегетарианское": veg_judge.invoke("Блюдо, которое предложил ассистент:\n" + a).vegetarian,
                     "мясо по словарю": MEAT.search(a) is not None})
veg = pd.DataFrame(rows)
res = veg.groupby("память", sort=False).agg(вегетарианских=("вегетарианское", "sum"),
                                             с_мясом_по_словарю=("мясо по словарю", "sum"), всего=("ответ", "size"))
print(res.to_string())
print(f"судья и словарь согласны в {(veg['вегетарианское'] != veg['мясо по словарю']).mean():.0%} ответов "
      f"(словарь грубый: «без мяса» он тоже подсветит)")
v = res["вегетарианских"]
print("вывод:", "память фактов чаще остальных предлагает вегетарианское" if v["память фактов"] > v.drop("память фактов").max()
      else "память фактов не обошла остальные — читаем ответы")
for r in veg[veg.память == "память фактов"].itertuples():
    dish_card(f"память фактов · {r.вопрос}", r.ответ[:300])

Сверяем с ожиданием по строке «вывод». Всё вегетарианское у буфера — случайность: фразы Паши он не видел. У всей истории
каждое «вегетарианское» — это факт, который модель заметила среди курицы, а каждое мясное — факт, который она
пропустила. У памяти фактов вегетарианство — если его извлекли — лежит отдельной короткой строкой. Четыре вопроса — это ещё не статистика: разница
в один ответ — монетка, а вот три-четыре из четырёх видны и без интервалов. А если память фактов всё-таки предложила мясо, загляните в неё
(`facts_mem.read("")`): извлечь факт мало, его ещё нужно не потерять при консолидации.

**Загадка 2 — зачем память при окне в 262 тысячи?** Три ответа с цифрами из этой пары: длинный контекст **дорогой**
(квадратичный счёт за диалог, раздел 1), **медленный** (секунды до первого слова и гигабайты KV-кэша, раздел 2)
и **не гарантирует внимания** (загадка 1). Память — это не способ «влезть», это способ **выбрать**.

**Загадка 3 — «удалите мои данные».** Факт живёт везде, куда мы его записали: в истории, сводке, векторах, фактах,
ядре агента, а ещё в логах и бэкапах. Удаление надо проектировать с первого дня: ссылка на источник у каждой записи,
удаление по источнику, а не по ключевым словам.

## Итоги

- **У модели нет памяти.** Всё, что она «помнит», мы кладём в контекст сами, и наивный способ — вся история заново —
  стоит $\sim n^2$ токенов за диалог.
- **Длинный контекст ≠ память:** дорого, медленно, и факт в контексте ещё не значит, что модель его применит.
  Кэш префикса (если движок его умеет) экономит вычисления, но не внимание.
- **Краткосрочная память** (буфер, окно, сводка) дешёвая, но теряет прошлое и время.
- **Долговременная память** бывает семантической, эпизодической и процедурной (CoALA). Векторный поиск по репликам
  находит похожее, а не нужное; извлечённые факты в разы компактнее всей истории.
- **Консолидация** (ADD / UPDATE / DELETE / NOOP, как в Mem0) убирает противоречия, но стирает историю; интервалы
  действия (Zep / Graphiti) её сохраняют, а выборка по давности, важности и релевантности (Generative Agents) держит
  контекст коротким, когда фактов тысячи.
- **Записывать можно в фоне или прямо в диалоге.** Во втором случае модель ведёт память сама (MemGPT / Letta) — ценой
  вызова на каждую реплику и ошибок «что считать фактом».
- **Внутри модели** память — KV-кэш (растёт) или состояние линейного внимания (фиксировано, как у Gated DeltaNet
  в Qwen3.5); она живёт один запрос.
- **Видео и камера** — та же идея: кадры → подписи с таймкодами → поиск. Это персональные данные, и удалять их надо
  по источнику, вместе со всем, что из него выведено.

## Попробуйте сами

- Добавьте в стратегию «факты + время» бюджет: не больше 300 токенов памяти. Где на графике Парето окажется точка?
- Замените в MemGPT модель на `qwen3.5:9b`. Насколько вырастет доля записанных фактов и упадёт доля ложных записей?
- Снимите на телефон минуту своего стола, задайте 5 вопросов «что было, когда появилось» и сравните «один кадр» с
  «поиском + пересмотром».

## Литература

- Liu et al. *Lost in the Middle: How Language Models Use Long Contexts*, 2023.
- Sumers, Yao, Narasimhan, Griffiths. *Cognitive Architectures for Language Agents* (CoALA), 2023.
- Park et al. *Generative Agents: Interactive Simulacra of Human Behavior*, 2023.
- Packer et al. *MemGPT: Towards LLMs as Operating Systems*, 2023.
- Chhikara et al. *Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory*, 2025.
- Rasmussen et al. *Zep: A Temporal Knowledge Graph Architecture for Agent Memory*, 2025.
- Maharana et al. *Evaluating Very Long-Term Conversational Memory of LLM Agents* (LoCoMo), 2024.
- Wu et al. *LongMemEval: Benchmarking Chat Assistants on Long-Term Interactive Memory*, 2024.
- Schlag, Irie, Schmidhuber. *Linear Transformers Are Secretly Fast Weight Programmers*, 2021 — дельта-правило.
- Yang, Kautz, Hatamizadeh. *Gated Delta Networks: Improving Mamba2 with Delta Rule*, 2024.
- Behrouz, Zhong, Mirrokni. *Titans: Learning to Memorize at Test Time*, 2025.
- Mu, Li, Goodman. *Learning to Compress Prompts with Gist Tokens*, 2023.
- Song et al. *MovieChat: From Dense Token to Sparse Memory for Long Video Understanding*, 2024.
- He et al. *MA-LMM: Memory-Augmented Large Multimodal Model for Long-Term Video Understanding*, 2024.
- Wang, Zhang, Zohar, Yeung-Levy. *VideoAgent: Long-form Video Understanding with Large Language Model as Agent*, 2024.
- Fan et al. *VideoAgent: A Memory-augmented Multimodal Agent for Video Understanding*, 2024.

Видео: «Baking bread at home», автор Arquus, конвертация в OGV — Leonel Sohns, Wikimedia Commons
(https://commons.wikimedia.org/wiki/File:Baking_bread_at_home.ogv), лицензия CC BY-SA 4.0
(https://creativecommons.org/licenses/by-sa/4.0/).